# Cartographie sémantique Mars (PetCare) — pipeline v2

Ce notebook refait le notebook de l'équipe (`mars_full_pipeline.ipynb`) en plus court, en français, avec les mêmes résultats.
Les étapes 1 (téléchargement) et 2 (vectorisation bge-m3) sont **gelées** : on lit leurs résultats dans les zips.

**Ce qu'il produit** : des thèmes (K-means) pour les 658 articles, leur nom, quatre axes Mars (animal, type d'aliment, stade de vie, besoin santé),
une carte 2D interactive, puis (étapes suivantes) la cannibalisation et les espaces vides.

**Comment l'exécuter** : `Exécuter tout`. Les paramètres sont tous dans la section 1.
Fichiers attendus : `mars_step1_data.zip`, `mars_step2_embeddings.zip` (obligatoires), `mars_step3_d3_eval.zip` (optionnel).

## 0. Installation et chemins
**Ce que ça fait** : n'installe que les paquets qui manquent (en local avec le venv du projet : aucun ; sur Colab : quelques-uns), puis fixe les dossiers. En local, choisir le noyau (kernel) du venv `.venv`, sinon Python cherche à tout installer.
**Pourquoi** : un seul endroit pour les chemins, local comme Colab. En local les zips sont dans `data/`, sur Colab dans `/content` (ou `/content/data`, ou le Drive).

In [ ]:
import importlib.util, subprocess, sys
PAQUETS = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "umap": "umap-learn", "plotly": "plotly", "sentence_transformers": "sentence-transformers"}   # module Python -> nom du paquet à installer
manquants = [paquet for module, paquet in PAQUETS.items() if importlib.util.find_spec(module) is None]
if manquants:                      # on n'installe que ce qui manque (en local avec le venv : rien ; sur Colab : peu de choses)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
print("paquets installés à l'instant :", manquants or "aucun, tout était déjà là")

import os, sys
IN_COLAB = "google.colab" in sys.modules
RACINE = "." if os.path.isdir("data") else ".."        # le notebook peut être lancé depuis notebooks/
DATA_DIR = "/content/data" if IN_COLAB else f"{RACINE}/data"
OUT_DIR = "/content/outputs" if IN_COLAB else f"{RACINE}/outputs"
DOSSIERS_ZIP = [DATA_DIR, "/content", "/content/drive/MyDrive"]   # où chercher les zips
os.makedirs(OUT_DIR, exist_ok=True)
print("Colab" if IN_COLAB else "Local", "| Python :", sys.executable, "| données :", DATA_DIR, "| sorties :", OUT_DIR)

**Imports** : tout ce que le calcul utilise. `sentence-transformers` (et `torch`) ne servent qu'à encoder quelques textes avec bge-m3
(noms de thèmes, familles, axes, articles espagnols). `scipy` sert au contour des zones de la carte.

In [ ]:
import io, glob, re, json, zipfile, html
import numpy as np
import pandas as pd
import umap
from plotly.offline import get_plotlyjs
from scipy.spatial import ConvexHull
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.manifold import trustworthiness
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_samples
from sklearn.preprocessing import normalize

## 1. Paramètres
Tout ce qu'on peut régler est ici, avec son rôle et l'effet d'un changement. Rien de ce qui suit n'a de valeur « cachée ».

**Paramètres généraux** : graine, nombre de thèmes, réglages du K-means, options.

In [ ]:
GRAINE = 42        # graine unique (K-means, UMAP) : la changer change légèrement les résultats
K = 15             # nombre de thèmes, fixé à la main (comme l'original) : plus grand = thèmes plus fins
N_INIT = 20        # K-means : nombre de départs aléatoires, on garde le meilleur
MAX_ITER = 300     # K-means : itérations maximum par départ
MODELE = "BAAI/bge-m3"   # modèle de vectorisation : le même qu'à l'étape 2, sinon les vecteurs ne sont plus comparables
AUTRES_LANGUES = True    # True : les 4 articles espagnols écartés à l'étape 1 sont vectorisés et classés (662 articles) ; False : 658
UTILISER_D3 = True       # True : étiquettes d3-turbo si le zip est présent (sinon méthode maison seule : résultats différents)

**Animal (chien / chat)** : on compte les mots du titre et de l'URL (comptés 3 fois) et du texte (1 fois).

In [ ]:
MOTS_CHIEN = {"dog", "dogs", "puppy", "puppies", "pup", "pups", "canine", "canines", "doggy", "doggie",
              "perro", "perros", "cachorro", "cachorros"}                    # mots qui désignent un chien
MOTS_CHAT = {"cat", "cats", "kitten", "kittens", "kitty", "feline", "felines", "gato", "gatos", "gatito", "gatitos"}
SEUIL_ESPECE = 0.80        # >= 80 % des mots pour une espèce -> "dog" ou "cat", sinon "dog and cat" (bas = plus de mixtes)
SEUIL_ESPECE_THEME = 0.60  # un thème a l'espèce d'au moins 60 % de ses articles, sinon "mixed"

**Noms de thèmes** : expressions candidates, choisies par le sens (bge-m3) et la distinction avec les autres thèmes.

In [ ]:
N_CANDIDATS = 80        # expressions candidates gardées par thème (plus = plus de choix, plus lent)
COUVERTURE_MIN = 0.15   # une candidate doit apparaître dans >= 15 % des articles du thème
POIDS_DISTINCTION = 0.5 # pénalité si l'expression ressemble aussi à un autre thème (haut = noms plus distincts)
POIDS_COUVERTURE = 0.15 # bonus si l'expression est dans beaucoup d'articles du thème (haut = noms plus représentatifs)
N_MOTS_CLES = 8         # mots-clés (tableau des thèmes, nommage des sous-thèmes santé si besoin)
N_TITRES = 10           # nombre de titres « typiques » listés par thème

**Mots à ignorer** pour construire les noms de thèmes (mots vides, noms de marques, mots trop généraux).

In [ ]:
MOTS_VIDES = list(ENGLISH_STOP_WORDS | {   # ignorés partout : marques, verbes creux, noms d'auteurs
    "iams", "eukanuba", "pedigree", "greenies", "cesar", "mars", "also", "may", "can", "will", "one",
    "help", "make", "like", "need", "get", "way", "time", "good", "best", "know", "important", "new",
    "great", "use", "keep", "sure", "day", "days", "want", "lot", "just", "really", "things", "thing",
    "don", "doesn", "isn", "ll", "ve", "re", "kelley", "russ", "sample"})
MOTS_GENERAUX = {"dog", "dogs", "cat", "cats", "puppy", "puppies", "kitten", "kittens", "pet", "pets", "food",
                 "foods", "health", "healthy", "owner", "owners", "life", "home", "article", "tips", "guide",
                 "help", "helps", "helping", "bring", "brings", "cause", "causes", "reason", "reasons", "need",
                 "needs", "care", "caring", "way", "ways", "sign", "signs", "fact", "facts", "benefit", "benefits",
                 "animal", "animals", "pup", "pups", "common", "different", "right", "better", "start", "come",
                 "says", "work", "run", "people", "family", "year", "years", "old", "little", "big"}
# MOTS_GENERAUX : interdits comme nom de thème s'ils sont seuls (trop présents dans un corpus petcare)

**Familles Mars** : chaque thème et chaque article est rattaché à la famille la plus proche en sens.
Chaque entrée : `nom affiché: (nom encodé, description)`. Le texte réellement encodé est « nom encodé: description ».
Les noms encodés sont ceux du run validé du 27/10 (en français) : les garder donne exactement les mêmes familles.

In [ ]:
FAMILLES = {
    "Nutrition & feeding": ("Nutrition & alimentation",
        "pet food, nutrients, ingredients, diet, how much and how often to feed, wet and dry food"),
    "Health & care": ("Santé & soins",
        "pet health problems, diseases, symptoms, digestion, vomiting, allergies, vaccines, vet visits, teeth, "
        "dental care, bad breath, oral health, dental chews, grooming, bathing, brushing, skin and coat care, hygiene"),
    "Education & training": ("Éducation & dressage",
        "training a dog or cat, obedience, commands, house training, crate training, leash training"),
    "Behaviour": ("Comportement",
        "pet behaviour and emotions, why pets do things, anxiety, barking, meowing, body language"),
    "Adoption & first days": ("Adoption & premiers jours",
        "adopting a pet, choosing a breed, rescue and shelters, bringing a new puppy or kitten home"),
    "Sport & activity": ("Sport & activité",
        "exercise, sporting and hunting dogs, agility, running with a dog, outdoor activities, performance"),
    "Travel & daily life": ("Voyage & vie quotidienne",
        "travelling with a pet, holidays and seasons, safety at home, lifestyle and events with pets"),
}
TEXTES_FAMILLES = [f"{nom_encode}: {description}" for nom_encode, description in FAMILLES.values()]

**Axes Mars** : pour chaque étiquette, des mots-clés (comptés dans le titre/URL ×3 et le texte) et une description (sens bge-m3).
Un article reçoit l'étiquette si : mots ≥ `SEUIL_MOTS_FORT`, ou mots ≥ `SEUIL_MOTS_MIN` et sens ≥ `Z_CONFIRME`, ou sens ≥ `Z_SEUL`.

In [ ]:
SEUIL_MOTS_FORT = 8   # assez de mots-clés à eux seuls (bas = plus d'étiquettes)
SEUIL_MOTS_MIN = 2    # quelques mots, à confirmer par le sens
Z_CONFIRME = 0.5      # sens au-dessus de la moyenne du corpus (en écarts-types)
Z_SEUL = 1.5          # sens très fort, même sans mots-clés
K_SANTE = 6           # nombre de sous-thèmes parmi les articles « besoin santé = oui »
AXE_ALIMENT = {"defaut": "not food-specific", "etiquettes": {     # étiquette: (mots-clés, description)
    "dry": (["dry food", "dry dog food", "dry cat food", "dry diet", "kibble", "kibbles", "croquette",
             "croquettes", "crunchy food"], "dry pet food, kibble, crunchy dry dog food or dry cat food"),
    "wet": (["wet food", "wet dog food", "wet cat food", "wet diet", "canned food", "canned", "pouch",
             "pouches", "pate", "pâté", "gravy", "jelly", "loaf", "moist food", "chunks"],
            "wet pet food, canned food, pouches, pâté, chunks in gravy or jelly"),
    "treats": (["treat", "treats", "chew", "chews", "dental chew", "dental chews", "dental treat",
                "dental treats", "snack", "snacks", "biscuit", "biscuits", "dentastix"],
               "pet treats, dental chews, snacks and biscuits given between meals")}}

AXE_STADE = {"defaut": "all ages", "etiquettes": {
    "puppy / kitten": (["puppy", "puppies", "kitten", "kittens", "junior", "growth", "growing", "weaning",
                        "newborn"], "puppies and kittens, young pets in their first months, growth and weaning"),
    "adult": (["adult", "adults", "adulthood", "adult dog", "adult cat"],
              "adult dogs and adult cats, everyday maintenance of grown pets"),
    "senior": (["senior", "seniors", "older dog", "older dogs", "older cat", "older cats", "aging", "ageing",
                "elderly", "mature", "golden years"],
               "senior and older pets, ageing dogs and cats over seven years old")}}
AXE_SANTE = {"defaut": "no", "etiquettes": {
    "yes": (["disease", "diseases", "symptom", "symptoms", "illness", "vet", "veterinarian", "digestion",
             "digestive", "diarrhea", "vomiting", "allergy", "allergies", "arthritis", "joint", "joints",
             "kidney", "urinary", "diabetes", "obesity", "overweight", "weight loss", "skin", "itching",
             "dental", "teeth", "gums", "plaque", "tartar", "bad breath", "infection", "parasite",
             "parasites", "flea", "fleas", "tick", "ticks", "sensitive stomach"],
            "pet health needs and conditions: digestion, weight, joints, skin and coat, teeth and oral "
            "health, kidney and urinary problems, allergies, symptoms and veterinary care")}}
AXES_MARS = {"aliment": AXE_ALIMENT, "stade": AXE_STADE, "sante": AXE_SANTE}

**d3-turbo** (optionnel) : pour chaque étiquette, la méthode retenue par l'équipe après comparaison sur 80 articles lus à la main le 27/10 (`d3` = étiquettes d3-turbo, `nous` = mots-clés + bge-m3).
Si le zip d3 est absent, tout passe en `nous` et **les résultats changent** (le notebook le dit).

In [ ]:
MEILLEURE_SOURCE = {"dry": "d3", "wet": "d3", "treats": "d3", "puppy / kitten": "d3", "senior": "d3",
                    "adult": "d3_choix",   # la question oui/non est peu fiable pour « adulte » : on utilise la question à choix unique
                    "health": "d3"}         # l'animal (chien/chat) reste le comptage de mots
NOM_D3 = {"dry": "sec", "wet": "humide", "treats": "friandises", "puppy / kitten": "chiot / chaton",
          "adult": "adulte", "senior": "senior", "health": "sante"}   # noms des étiquettes dans le fichier d3

**Carte 2D (UMAP)** : réglages de la projection et des zones colorées.

In [ ]:
UMAP_VOISINS = 15       # UMAP : voisins considérés (petit = structure locale, grand = vue d'ensemble)
UMAP_MIN_DIST = 0.1     # UMAP : écart minimal entre points (petit = groupes plus serrés)
N_VOISINS_CONFIANCE = 10  # voisins utilisés pour la « fidélité » de la carte (trustworthiness, 1 = parfait)
PART_ZONE = 0.75        # part des articles d'un thème (les plus centraux) dans sa zone colorée
ZONE_MAX = 0.08         # pas de zone si elle couvre plus de 8 % de la carte (thème trop dispersé)

**Étape 4 — cannibalisation et espaces vides** : réglages.

In [ ]:
SEUIL = 0.82             # cannibalisation : deux articles de marques différentes sont « en concurrence » si leur similarité est >= 0,82 (bas = plus de paires)
MIN_CANDIDAT = 0.70      # sous cette similarité une paire n'est jamais étudiée (le tableau de sensibilité part de là)
MIN_AUTRES_LACUNE = 3    # « lacune » d'une marque : 0 article là où les autres marques en ont au moins 3
MIN_PERIMETRE = 3        # une marque « travaille » un animal ou un type d'aliment si elle a au moins 3 articles dessus
MIN_CURSEUR = 0.75       # tableau de bord : plus petit seuil proposé par le curseur (bas = plus de paires, page plus lourde ; doit être >= MIN_CANDIDAT)
MAX_CURSEUR = 0.95       # tableau de bord : plus grand seuil proposé par le curseur

## 2. Chargement des données gelées (étapes 1 et 2)
**Ce que ça fait** : lit les vecteurs, l'index et le texte des articles directement dans les zips (rien n'est décompressé).
**Règle critique** : la ligne *i* de `embeddings.npy` est l'article de la ligne *i* de `embeddings_index.csv`. On ne trie et on ne filtre jamais l'un sans l'autre.

In [ ]:
def ouvrir_zip(motif, obligatoire=True):
    """Ouvre le zip le plus récent qui correspond au motif, dans DOSSIERS_ZIP (None si absent et optionnel)."""
    trouves = [p for d in DOSSIERS_ZIP for p in glob.glob(f"{d}/{motif}")]
    if not trouves and obligatoire:
        raise FileNotFoundError(f"{motif} introuvable dans {DOSSIERS_ZIP}")
    return zipfile.ZipFile(max(trouves, key=os.path.getmtime)) if trouves else None

zip_vecteurs = ouvrir_zip("mars_step2_embeddings*.zip")
zip_corpus = ouvrir_zip("mars_step1_data*.zip")
vecteurs = np.load(io.BytesIO(zip_vecteurs.read("all/embeddings.npy"))).astype(np.float32)
index = pd.read_csv(zip_vecteurs.open("all/embeddings_index.csv"))
corpus = pd.read_csv(zip_corpus.open("all/corpus_clean.csv"))
print("vecteurs", vecteurs.shape, "| index", index.shape, "| corpus", corpus.shape)

**Contrôles d'alignement** puis ajout du texte de chaque article (jointure sur l'URL, sans changer l'ordre ni le nombre de lignes).
On renomme les colonnes en français : `marque`, `titre`, `texte`, `n_mots`.

In [ ]:
normes = np.linalg.norm(vecteurs, axis=1)
assert len(vecteurs) == len(index), "vecteurs et index n'ont pas le même nombre de lignes"
assert (index["row"].to_numpy() == np.arange(len(index))).all(), "la colonne row de l'index n'est pas 0..n-1"
assert 0.99 < normes.min() and normes.max() < 1.01, "vecteurs non normalisés"
print(f"normes : min {normes.min():.4f}, max {normes.max():.4f}")

textes = corpus[["url", "text_clean", "description", "lang"]].drop_duplicates("url")
index = index.merge(textes, on="url", how="left")
assert len(index) == len(vecteurs), "la jointure a changé le nombre de lignes"
index = index.rename(columns={"brand": "marque", "title": "titre", "text_clean": "texte", "n_words_clean": "n_mots"})
for col in ["texte", "titre", "description"]:
    index[col] = index[col].fillna("").astype(str)
index["langue"] = index["lang"].fillna("en").astype(str)
index[["marque", "titre", "n_mots", "langue"]].head(3)

**Articles non anglais** (paramètre `AUTRES_LANGUES`). L'étape 1 a écarté 4 pages espagnoles ; on les vectorise ici avec bge-m3 (même texte qu'à l'étape 2 :
titre + description + texte) pour les classer avec les autres. bge-m3 est multilingue : un article espagnol se place près des articles anglais du même sujet.
C'est ce qui fait passer de 658 à 662 articles. Les étiquettes `en` / `es` deviennent `English` / `Spanish`.

In [ ]:
modele = None
def encoder(textes):
    """Vecteurs bge-m3 normalisés de la liste de textes (le modèle est chargé une seule fois)."""
    global modele
    if modele is None:
        import torch
        from sentence_transformers import SentenceTransformer
        modele = SentenceTransformer(MODELE, device="cuda" if torch.cuda.is_available() else "cpu")
    return np.asarray(modele.encode(list(textes), normalize_embeddings=True, batch_size=64), dtype=np.float32)

def texte_pour_modele(r):
    """Même texte qu'à l'étape 2 : titre + description (si pas déjà dans le texte) + texte."""
    parties = [r["titre"].strip()]
    d = r["description"].strip()
    if d and d.lower() not in r["texte"][:len(d) + 50].lower():
        parties.append(d)
    parties.append(r["texte"].strip())
    return "\n".join(p for p in parties if p)

def ajouter_autres_langues(vecteurs, index, corpus):
    c = corpus.rename(columns={"brand": "marque", "title": "titre", "text_clean": "texte", "n_words_clean": "n_mots"})
    for col in ["titre", "description", "texte", "slug", "marque"]:
        c[col] = c[col].fillna("").astype(str)
    garder = (~c["lang"].isin(["en", "?"])) & (c["texte"].str.split().str.len() >= 100) & (~c["url"].isin(index["url"]))
    garder &= c["page_type"] == "article"
    autres = c[garder].drop_duplicates("url").reset_index(drop=True)
    if autres.empty:
        return vecteurs, index
    nouveaux = autres[["url", "titre", "description", "texte", "slug", "marque", "n_mots"]].copy()
    nouveaux["langue"] = autres["lang"]
    nouveaux_vecteurs = encoder([texte_pour_modele(r) for _, r in autres.iterrows()])
    index = pd.concat([index, nouveaux], ignore_index=True)
    print(f"+ {len(autres)} articles non anglais :", autres["lang"].value_counts().to_dict())
    return np.vstack([vecteurs, nouveaux_vecteurs]), index

if AUTRES_LANGUES:
    vecteurs, index = ajouter_autres_langues(vecteurs, index, corpus)
index["langue"] = index["langue"].map({"en": "English", "es": "Spanish"}).fillna(index["langue"])
print("vecteurs", vecteurs.shape, "| index", index.shape, "|", index["langue"].value_counts().to_dict())

**Préparation des vecteurs** : on renormalise (norme 1) pour que le K-means euclidien équivaille à un regroupement par cosinus.
On vérifie aussi la médiane des similarités et les quasi-doublons (similarité > 0,99), à connaître avant d'interpréter la carte.

In [ ]:
X = vecteurs / np.clip(np.linalg.norm(vecteurs, axis=1, keepdims=True), 1e-12, None)
n = len(X)
sim = X @ X.T
haut = np.triu_indices(n, 1)
print(f"X : {X.shape} | similarité médiane entre deux articles : {np.median(sim[haut]):.3f}")
doublons = np.argwhere(np.triu(sim, 1) > 0.99)
print(f"{len(doublons)} paires quasi identiques (> 0,99) :")
for a, b in doublons:
    print("  ", index.at[a, "titre"][:50], "|", index.at[b, "titre"][:50])

## 3. Thèmes (K-means)
**Ce que ça fait** : regroupe les articles en `K` thèmes. **Pourquoi** : c'est la base de la carte, des noms et des espaces vides.
**Paramètres qui comptent** : `K`, `GRAINE`, `N_INIT`. Pour chaque article on garde aussi la silhouette (1 = bien rangé, 0 = à la frontière)
et le thème voisin le plus proche (utile pour repérer les articles « ponts »).

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=N_INIT, max_iter=MAX_ITER, random_state=GRAINE).fit(X)
theme = kmeans.labels_
centres = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, axis=1, keepdims=True)
distance_centre = 1 - np.sum(X * centres[theme], axis=1)      # 1 - cosinus entre l'article et son centre
sims_centres = X @ centres.T
sims_centres[np.arange(n), theme] = -np.inf                   # on exclut son propre thème
theme_voisin = sims_centres.argmax(axis=1)
silhouette = silhouette_samples(X, theme, metric="cosine")
print("articles par thème :", np.bincount(theme).tolist())
print(f"silhouette moyenne : {silhouette.mean():.3f} (proche de 0 = thèmes qui se chevauchent beaucoup)")

## 4. Noms des thèmes
**Idée** : nommer un thème avec les 2 expressions (1 à 3 mots) qui le résument le mieux et le distinguent des autres.
1. *Candidates* : expressions présentes dans au moins `COUVERTURE_MIN` des articles du thème, classées par fréquence (c-TF-IDF).
2. *Score* = ressemblance (bge-m3) avec le centre du thème − `POIDS_DISTINCTION` × ressemblance avec le thème voisin + `POIDS_COUVERTURE` × couverture.
3. La 2e expression doit être différente de la 1re (critère MMR, pas la même racine).

In [ ]:
VECTEUR_MOTS = dict(token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b", stop_words=MOTS_VIDES)

def mots_cles_par_theme(textes, labels, k):
    """c-TF-IDF : un « document » par thème = concaténation de ses articles. Renvoie N_MOTS_CLES mots par thème."""
    docs = [" ".join(textes[labels == c]) for c in range(k)]
    vect = CountVectorizer(ngram_range=(1, 2), min_df=1, **VECTEUR_MOTS)
    comptes = vect.fit_transform(docs).toarray().astype(float)
    tf = comptes / np.clip(comptes.sum(axis=1, keepdims=True), 1, None)
    idf = np.log(1 + (comptes.sum() / k) / np.clip(comptes.sum(axis=0), 1, None))
    vocab = np.array(vect.get_feature_names_out())
    return [vocab[np.argsort(-(tf * idf)[c])[:N_MOTS_CLES]].tolist() for c in range(k)]

def candidates_par_theme(textes, labels, k):
    """Expressions de 1 à 3 mots assez présentes dans le thème : [(expression, couverture)] par thème."""
    vect = CountVectorizer(ngram_range=(1, 3), min_df=2, binary=True, **VECTEUR_MOTS)
    presence = vect.fit_transform(textes)                                  # article x expression (0/1)
    vocab = np.array(vect.get_feature_names_out())
    par_theme = np.vstack([np.asarray(presence[labels == c].sum(axis=0)).ravel() for c in range(k)])
    couverture = par_theme / np.bincount(labels, minlength=k)[:, None]     # part des articles du thème
    idf = np.log(1 + par_theme.sum() / k / np.clip(par_theme.sum(axis=0), 1, None))
    general = np.array([set(e.split()) <= MOTS_GENERAUX for e in vocab])
    score = couverture * idf
    resultat = []
    for c in range(k):
        ok = np.where((couverture[c] >= COUVERTURE_MIN) & ~general)[0]
        ordre = ok[np.argsort(-score[c, ok])][:N_CANDIDATS]
        resultat.append([(vocab[i], float(couverture[c, i])) for i in ordre])
    return resultat

def racines(expression):
    """Mots sans le pluriel simple (dogs -> dog), pour éviter « Dog · Dogs »."""
    return {m[:-1] if m.endswith("s") and len(m) > 3 else m for m in expression.lower().split()}

def centres_unitaires(Xc, labels, k):
    c = np.stack([Xc[labels == i].mean(axis=0) for i in range(k)])
    return c / np.linalg.norm(c, axis=1, keepdims=True)

def choisir_expressions(Xc, labels, k, candidates):
    """Pour chaque thème, les 2 expressions retenues (liste de listes)."""
    ctr = centres_unitaires(Xc, labels, k)
    uniques = sorted({e for liste in candidates for e, _ in liste})
    vec = dict(zip(uniques, encoder(uniques))) if uniques else {}
    choisies = []
    for c in range(k):
        liste = candidates[c]
        if not liste:
            choisies.append([]); continue
        V = np.stack([vec[e] for e, _ in liste])
        couv = np.array([cv for _, cv in liste])
        autres = np.delete(np.arange(k), c)
        voisin = (V @ ctr[autres].T).max(axis=1)                  # ressemblance avec l'autre thème le plus proche
        score = V @ ctr[c] - POIDS_DISTINCTION * voisin + POIDS_COUVERTURE * couv
        premier = int(np.argmax(score))
        mmr = 0.7 * score - 0.3 * (V @ V[premier])                # 2e : bon score ET différente de la 1re
        for i, (e, _) in enumerate(liste):
            if i == premier or racines(e) & racines(liste[premier][0]):
                mmr[i] = -np.inf
        choisies.append([liste[premier][0]] + ([liste[int(np.argmax(mmr))][0]] if np.isfinite(mmr).any() else []))
    return choisies

**Application** : textes = titre (compté 2 fois) + texte. Nom d'un thème = les 2 expressions en majuscule initiale, séparées par « · ».
On garde aussi les mots-clés (c-TF-IDF) pour le tableau des thèmes.

In [ ]:
textes_themes = (index["titre"] + " " + index["titre"] + " " + index["texte"]).to_numpy()
mots_cles = mots_cles_par_theme(textes_themes, theme, K)
expressions = choisir_expressions(X, theme, K, candidates_par_theme(textes_themes, theme, K))
theme_nom = [" · ".join(e.capitalize() for e in expr) or " / ".join(mots_cles[c][:3]) for c, expr in enumerate(expressions)]
pd.DataFrame({"theme": range(K), "nom": theme_nom, "n_articles": np.bincount(theme)})

## 5. Familles Mars
**Ce que ça fait** : rattache chaque **thème** à la famille Mars la plus proche de son centre, et chaque **article** à la famille la plus proche de son vecteur.
**Pourquoi les deux** : un sujet rare et dispersé (ex. toilettage) peut être dans un thème d'une autre famille mais retrouver sa propre famille au niveau article.

In [ ]:
noms_familles = list(FAMILLES)
vecteurs_familles = encoder(TEXTES_FAMILLES)
sim_theme_famille = centres_unitaires(X, theme, K) @ vecteurs_familles.T
famille_theme = [noms_familles[i] for i in sim_theme_famille.argmax(axis=1)]
famille_theme_score = sim_theme_famille.max(axis=1)
famille_article = [noms_familles[i] for i in (X @ vecteurs_familles.T).argmax(axis=1)]
print(pd.Series(famille_article).value_counts().to_dict())
pd.DataFrame({"theme": range(K), "famille": famille_theme}).head(5)

## 6. Animal : chien, chat ou les deux
**Ce que ça fait** : compte les mots « chien » et « chat » (titre et URL ×3, texte ×1). Si une espèce a au moins `SEUIL_ESPECE` des mots → cette espèce ;
sinon « dog and cat » ; aucun mot → « unspecified ». Un thème prend l'espèce d'au moins `SEUIL_ESPECE_THEME` de ses articles, sinon « mixed ».

In [ ]:
def compter(texte, mots):
    return sum(1 for m in re.findall(r"[a-záéíóúñ]+", texte.lower()) if m in mots)

def espece_article(r):
    tete = r["titre"] + " " + str(r["url"]).replace("-", " ").replace("/", " ")
    n_chien = 3 * compter(tete, MOTS_CHIEN) + compter(r["texte"], MOTS_CHIEN)
    n_chat = 3 * compter(tete, MOTS_CHAT) + compter(r["texte"], MOTS_CHAT)
    if n_chien + n_chat == 0:
        return "unspecified"
    if n_chien / (n_chien + n_chat) >= SEUIL_ESPECE:
        return "dog"
    return "cat" if n_chat / (n_chien + n_chat) >= SEUIL_ESPECE else "dog and cat"

espece = index.apply(espece_article, axis=1).tolist()
espece_theme = {}                                  # thème -> (espèce, part en %)
for c, g in pd.DataFrame({"c": theme, "e": espece}).groupby("c"):
    parts = g["e"].value_counts(normalize=True)
    espece_theme[int(c)] = (parts.index[0] if parts.iloc[0] >= SEUIL_ESPECE_THEME else "mixed", round(100 * parts.iloc[0], 1))
print(pd.Series(espece).value_counts().to_dict())

## 7. Axes Mars : type d'aliment, stade de vie, besoin santé
**Ce que ça fait** : pour chaque étiquette (ex. `dry`), on combine deux signaux : le nombre de mots-clés et le *sens* (similarité bge-m3 entre l'article et la description de l'étiquette,
centrée-réduite sur le corpus). Un article peut avoir plusieurs étiquettes ; l'étiquette principale est celle de meilleur score.
**Paramètres qui comptent** : `SEUIL_MOTS_FORT`, `SEUIL_MOTS_MIN`, `Z_CONFIRME`, `Z_SEUL`.

In [ ]:
def motif(mots):
    """Expression régulière qui trouve les mots/expressions de la liste (mots entiers)."""
    mots = sorted(mots, key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(re.escape(m).replace(r"\ ", r"\s+") for m in mots) + r")\b", re.I)

def etiquettes_axe(a, j0, Z, tete, corps):
    """Étiquettes (liste) et étiquette principale de chaque article pour un axe. j0 = 1re colonne de Z de l'axe."""
    etiq, principale = [[] for _ in range(n)], [None] * n
    meilleur = np.full(n, -np.inf)
    for j, (lab, (mots, _)) in enumerate(a["etiquettes"].items(), start=j0):
        rx = motif(mots)
        for i in range(n):
            m = 3 * len(rx.findall(tete[i])) + len(rx.findall(corps[i]))       # mots-clés : titre/URL x3 + texte
            z = float(Z[i, j])                                                  # sens : z-score bge-m3
            if m >= SEUIL_MOTS_FORT or (m >= SEUIL_MOTS_MIN and z >= Z_CONFIRME) or z >= Z_SEUL:
                etiq[i].append(lab)
                if z + m / SEUIL_MOTS_FORT > meilleur[i]:
                    meilleur[i], principale[i] = z + m / SEUIL_MOTS_FORT, lab
    return etiq, principale

tete = (index["titre"] + " " + index["url"].astype(str).str.replace(r"[-/_.]", " ", regex=True)).tolist()
corps = index["texte"].tolist()
descriptions = [d for a in AXES_MARS.values() for _, d in a["etiquettes"].values()]
S = X @ encoder(descriptions).T
Z = (S - S.mean(axis=0)) / np.clip(S.std(axis=0), 1e-6, None)          # sens centré-réduit sur le corpus

axes = pd.DataFrame(index=range(n))
j0 = 0
for nom_axe, a in AXES_MARS.items():
    etiq, principale = etiquettes_axe(a, j0, Z, tete, corps)
    axes[nom_axe] = [" + ".join(e) if e else a["defaut"] for e in etiq]
    axes[nom_axe + "_principal"] = [p or a["defaut"] for p in principale]
    j0 += len(a["etiquettes"])
print({c: axes[c].value_counts().to_dict() for c in ["aliment_principal", "stade_principal", "sante"]})

## 8. Étiquettes d3-turbo (optionnel)
**Ce que ça fait** : si le zip `mars_step3_d3_eval.zip` est présent, remplace, étiquette par étiquette, la méthode maison par d3-turbo là où il est meilleur (`MEILLEURE_SOURCE`).
**Attention** : sans ce zip, le notebook continue mais les axes aliment / stade / santé sont ceux de la méthode maison seule, donc **différents**.

In [ ]:
def appliquer_d3(axes, index, d3):
    """Remplace les étiquettes par celles de d3-turbo selon MEILLEURE_SOURCE (articles trouvés par URL)."""
    d3 = d3.drop_duplicates("url").set_index("url")
    choix_ok = "d3_stade_choix" in d3.columns
    axes = axes.copy()
    for i in axes.index:
        if index.at[i, "url"] not in d3.index:
            continue
        r = d3.loc[index.at[i, "url"]]
        for nom_axe, a in AXES_MARS.items():
            nos = set(axes.at[i, nom_axe].split(" + ")) - {a["defaut"]}
            retenues, probas = [], {}
            for lab in a["etiquettes"]:
                cle = "health" if nom_axe == "sante" else lab
                source = MEILLEURE_SOURCE.get(cle, "nous")
                if source == "d3_choix" and not choix_ok:
                    source = "nous"
                if source == "d3":
                    oui, p = bool(r[f"d3_{NOM_D3[cle]}"]), float(r[f"d3_p_{NOM_D3[cle]}"])
                elif source == "d3_choix":
                    oui, p = r["d3_stade_choix"] == NOM_D3[cle], float(r.get(f"d3_p_choix_{NOM_D3[cle]}", 0) or 0)
                else:
                    oui, p = lab in nos, None
                if oui:
                    retenues.append(lab)
                    probas[lab] = p if p is not None else 0.5
            axes.at[i, nom_axe] = " + ".join(retenues) if retenues else a["defaut"]
            axes.at[i, nom_axe + "_principal"] = max(probas, key=probas.get) if probas else a["defaut"]
    return axes

zip_d3 = ouvrir_zip("mars_step3_d3_eval*.zip", obligatoire=False) if UTILISER_D3 else None
if zip_d3 is not None:
    axes = appliquer_d3(axes, index, pd.read_csv(zip_d3.open("d3_etiquettes.csv")))
    source_axes = "d3-turbo (aliment, stade, santé) + comptage de mots (animal)"
else:
    source_axes = "méthode maison seule (mots-clés + bge-m3)"
    print("⚠️ d3-turbo absent ou désactivé : axes par la méthode maison seule, résultats différents.")
print("Source des axes :", source_axes)
print({c: axes[c].value_counts().to_dict() for c in ["aliment_principal", "stade_principal", "sante"]})

## 9. Sous-thèmes santé
**Ce que ça fait** : parmi les articles « besoin santé = oui », un second K-means (`K_SANTE` groupes) nommé comme les thèmes.
**Pourquoi** : « santé » est trop large pour agir ; les sous-thèmes (dents, poids, digestion…) montrent quoi traiter.

In [ ]:
def sous_themes_sante(sante):
    """Renvoie (colonne « sous-thème » pour tous les articles, tableau résumé)."""
    idx = np.where(np.asarray(sante) == "yes")[0]
    colonne = [""] * n
    if len(idx) < 30:
        return colonne, pd.DataFrame()
    k = max(2, min(K_SANTE, len(idx) // 8))
    lab = KMeans(n_clusters=k, n_init=N_INIT, random_state=GRAINE).fit(X[idx]).labels_
    sous = index.iloc[idx].reset_index(drop=True)
    txt = (sous["titre"] + " " + sous["titre"] + " " + sous["texte"]).to_numpy()
    expr = choisir_expressions(X[idx], lab, k, candidates_par_theme(txt, lab, k))
    noms = [" · ".join(e.capitalize() for e in x) or f"S{j}" for j, x in enumerate(expr)]
    lignes = []
    for j in range(k):
        membres = np.where(lab == j)[0]
        lignes.append({"sous_theme": f"S{j}", "nom": noms[j], "n_articles": len(membres),
                       "articles_par_marque": ", ".join(f"{m} {c}" for m, c in sous.loc[membres, "marque"].value_counts().items())})
        for m in membres:
            colonne[idx[m]] = f"S{j} · {noms[j]}"
    return colonne, pd.DataFrame(lignes)

sante_sous_theme, resume_sante = sous_themes_sante(axes["sante"].values)
resume_sante

## 10. Carte 2D (UMAP)
**Ce que ça fait** : projette les vecteurs en 2 dimensions avec UMAP (distance cosinus). **Attention** : une carte 2D déforme les distances ;
ne jamais conclure à une lacune de contenu à partir d'une zone vide. La *fidélité* (trustworthiness, de 0 à 1) mesure la part des voisins de la carte
qui sont de vrais voisins dans l'espace complet.
**Paramètres qui comptent** : `UMAP_VOISINS`, `UMAP_MIN_DIST`, `GRAINE`. `n_jobs=1` est nécessaire pour un résultat reproductible.

In [ ]:
coords = umap.UMAP(n_components=2, metric="cosine", n_neighbors=min(UMAP_VOISINS, n - 1),
                   min_dist=UMAP_MIN_DIST, random_state=GRAINE, n_jobs=1).fit_transform(X)
fidelite = round(float(trustworthiness(X, coords, n_neighbors=min(N_VOISINS_CONFIANCE, n // 2 - 1), metric="cosine")), 3)
print("coords", coords.shape, "| fidélité UMAP :", fidelite)

## 11. Tableau final : un article = une ligne
**Ce que ça fait** : rassemble tout dans `articles`, la table que les étapes suivantes (cannibalisation, espaces vides) réutilisent.

In [ ]:
articles = pd.DataFrame({
    "article_id": range(n), "url": index["url"], "marque": index["marque"], "langue": index["langue"],
    "titre": index["titre"], "slug": index["slug"], "n_mots": index["n_mots"],
    "theme": theme, "theme_nom": [theme_nom[c] for c in theme],
    "famille_theme": [famille_theme[c] for c in theme], "famille_article": famille_article,
    "espece": espece,
    "espece_theme": [f"{espece_theme[c][0]} ({espece_theme[c][1]:.0f}% of the articles)" for c in theme],
})
for col in axes.columns:
    articles[col] = axes[col].values
articles["source_axes"] = source_axes
articles["sante_sous_theme"] = sante_sous_theme
articles["theme_voisin"] = theme_voisin
articles["voisin_nom"] = [theme_nom[c] for c in theme_voisin]
articles["silhouette"] = silhouette.round(4)
articles["distance_centre"] = distance_centre.round(4)
articles["rang_dans_theme"] = articles.groupby("theme")["distance_centre"].rank(method="first").astype(int)
articles["umap_x"], articles["umap_y"] = coords[:, 0].round(4), coords[:, 1].round(4)
articles["description"] = index["description"].str.slice(0, 400)
print(articles.shape)
articles.drop(columns=["url", "slug", "description"]).head(3)

**Tableau des thèmes** (une ligne par thème : nom, famille, espèce, taille, titre représentatif, mots-clés) et **grille Mars**
(nombre d'articles par combinaison animal × aliment × stade × santé et par marque ; 0 = espace vide possible).

In [ ]:
lignes = []
for c in range(K):
    membres = np.where(theme == c)[0]
    proches = membres[np.argsort(-(X[membres] @ centres[c]))][:N_TITRES]       # les plus centraux
    anglais = [i for i in proches if index.at[i, "langue"] == "English"] or list(proches)
    lignes.append({"theme": c, "nom": theme_nom[c], "famille": famille_theme[c], "famille_score": round(float(famille_theme_score[c]), 3),
                   "espece": espece_theme[c][0], "espece_part_pct": espece_theme[c][1], "n_articles": len(membres),
                   "part_pct": round(100 * len(membres) / n, 1), "silhouette_moyenne": round(float(silhouette[membres].mean()), 3),
                   "titre_representatif": index.at[anglais[0], "titre"], "mots_cles": ", ".join(mots_cles[c]),
                   "titres_types": " | ".join(index.loc[proches, "titre"]),
                   "articles_par_marque": ", ".join(f"{m} {k}" for m, k in index.loc[membres, "marque"].value_counts().items())})
resume_themes = pd.DataFrame(lignes)
print(resume_themes.shape)
resume_themes[["theme", "nom", "famille", "n_articles", "titre_representatif"]]

def grille_mars(articles):
    """Articles par (animal, aliment, stade, santé) x marque, combinaisons vides incluses (0 = espace vide possible)."""
    d = articles[["marque", "espece", "aliment", "stade", "sante"]].copy()
    for axe in ("aliment", "stade"):                              # un article à plusieurs étiquettes compte dans chacune
        d[axe] = d[axe].str.split(" + ", regex=False)
        d = d.explode(axe)
    valeurs = {"espece": ["dog", "cat", "dog and cat", "unspecified"],
               "aliment": list(AXE_ALIMENT["etiquettes"]) + [AXE_ALIMENT["defaut"]],
               "stade": list(AXE_STADE["etiquettes"]) + [AXE_STADE["defaut"]], "sante": ["yes", "no"]}
    g = d.groupby(list(valeurs) + ["marque"]).size().unstack("marque", fill_value=0)
    g = g.reindex(pd.MultiIndex.from_product(list(valeurs.values()), names=list(valeurs)), fill_value=0)
    g["TOTAL"] = g.sum(axis=1)
    return g

grille = grille_mars(articles)
couverture = pd.crosstab([articles["famille_theme"], articles["theme"].astype(str) + " · " + articles["theme_nom"]],
                         articles["marque"], rownames=["famille", "theme"], colnames=["marque"], margins=True, margins_name="TOTAL")
print("grille", grille.shape, "| couverture thèmes x marques", couverture.shape)
couverture.head(5)

**Enregistrement** des résultats de l'étape 3 dans le dossier de sorties (les étapes suivantes les réutilisent en mémoire).

In [ ]:
articles.drop(columns=["description"]).to_csv(f"{OUT_DIR}/articles_themes.csv", index=False, encoding="utf-8-sig")
resume_themes.to_csv(f"{OUT_DIR}/themes_resume.csv", index=False, encoding="utf-8-sig")
resume_sante.to_csv(f"{OUT_DIR}/sous_themes_sante.csv", index=False, encoding="utf-8-sig")
grille.to_csv(f"{OUT_DIR}/grille_mars.csv", encoding="utf-8-sig")
couverture.to_csv(f"{OUT_DIR}/couverture_marques_themes.csv", encoding="utf-8-sig")
print(sorted(os.listdir(OUT_DIR)))

---
# Étape 4 — Cannibalisation et espaces vides
**Cannibalisation** : deux articles de **marques différentes** très proches en sens se concurrencent sur le même sujet. On garde les paires dont la similarité dépasse un seuil (`SEUIL`), puis on les regroupe.
**Espaces vides** : combinaisons (animal × aliment × stade) ou thèmes où une marque n'a aucun article alors que les autres en ont.
**Attention** : une similarité élevée ne prouve pas la même intention de recherche, et un trou sur la carte 2D n'est pas une lacune : les chiffres ci-dessous sont des pistes à vérifier.

## 12. Vecteurs de l'étape 4
**Ce que ça fait** : reprend les **658 articles de l'étape 2** (les 4 articles espagnols n'ont pas de vecteurs de paragraphes, ils sont exclus des paires ; ils restent comptés dans les espaces vides, comme dans l'original),
et charge les vecteurs de paragraphes (2 961) pour mesurer si deux articles recouvrent les mêmes contenus, pas seulement le même sujet.

In [ ]:
n_etape2 = int(index["row"].notna().sum())               # articles de l'étape 2 (les espagnols n'ont pas de colonne row)
assert index["row"].iloc[:n_etape2].notna().all(), "les articles espagnols doivent venir à la fin"
art = index.iloc[:n_etape2].reset_index(drop=True)         # ligne i = vecteur i (règle d'alignement)
X4 = normalize(vecteurs[:n_etape2], norm="l2")
marques = sorted(art["marque"].unique())
marque_art = art["marque"].to_numpy(dtype=object)
vecteurs_para = normalize(np.load(io.BytesIO(zip_vecteurs.read("all/chunk_embeddings.npy"))), norm="l2")
para_de = pd.read_csv(zip_vecteurs.open("all/chunks.csv")).groupby("doc_row").indices   # article -> ses paragraphes
assert len(art) == len(X4) and max(para_de) < n_etape2
print("articles", art.shape, "| vecteurs", X4.shape, "| paragraphes", vecteurs_para.shape, "|", len(marques), "marques :", marques)

## 13. Paires candidates
**Ce que ça fait** : calcule la similarité cosinus de toutes les paires d'articles et garde celles ≥ `MIN_CANDIDAT`.
**Pourquoi un seuil** : la similarité médiane entre deux articles quelconques est de 0,575 ; presque tous les articles se ressemblent un peu, il faut donc ne garder que les paires **très** proches.

In [ ]:
S4 = X4 @ X4.T
iu, ju = np.triu_indices(n_etape2, 1)
sim4 = S4[iu, ju]
garde = sim4 >= MIN_CANDIDAT
paires = pd.DataFrame({"a": iu[garde], "b": ju[garde], "similarite": sim4[garde]})
paires["entre_marques"] = marque_art[paires["a"]] != marque_art[paires["b"]]
entre = paires[paires["entre_marques"]]
print(f"similarité médiane : {np.median(sim4):.3f} | {100 * (sim4 >= 0.5).mean():.0f} % des paires sont >= 0,50")
print(f"paires candidates (>= {MIN_CANDIDAT}) : {len(entre)} entre marques, {int((~paires['entre_marques']).sum())} dans une même marque")
paires.head(3)

## 14. Le seuil : pourquoi 0,82
**Ce que ça fait** : montre combien de paires on garde selon le seuil, puis des paires juste au-dessus et juste en dessous de `SEUIL` pour juger à l'œil si la coupure est raisonnable.
**Comment lire** : 0,80 garde beaucoup de paires (359), 0,85 très peu (75). 0,82 est un compromis simple : environ le **0,1 % des paires entre marques les plus proches**, assez pour dégager des cas nets
sans noyer la lecture. C'est un choix de lecture, pas une vérité : le paramètre `SEUIL` se change en une ligne et tout le reste se recalcule.

In [ ]:
assert SEUIL >= MIN_CANDIDAT
niveaux = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
sensibilite = pd.DataFrame([{"seuil": t, "paires_entre_marques": int((entre["similarite"] >= t).sum()),
                             "paires_meme_marque": int(((paires["similarite"] >= t) & ~paires["entre_marques"]).sum())}
                            for t in sorted(set(niveaux + [SEUIL]))])
print(f"seuil retenu : {SEUIL} (= {100 * (entre['similarite'] >= SEUIL).sum() / (marque_art[iu] != marque_art[ju]).sum():.2f} % de toutes les paires entre marques)")
sensibilite

proches = entre.assign(ecart=(entre["similarite"] - SEUIL).abs()).nsmallest(8, "ecart").sort_values("similarite")
for _, p in proches.iterrows():
    print(f"{'GARDÉE ' if p['similarite'] >= SEUIL else 'écartée'} {p['similarite']:.3f} | {art.at[p['a'], 'marque']}: {art.at[p['a'], 'titre'][:48]} | {art.at[p['b'], 'marque']}: {art.at[p['b'], 'titre'][:48]}")

## 15. Paires retenues et groupes
**Ce que ça fait** : garde les paires de marques différentes au-dessus du seuil, ajoute pour chaque article son thème et ses axes Mars (étape 3), puis les regroupe par **lien complet** :
dans un groupe, *tous* les articles sont proches deux à deux (≥ la plus petite similarité retenue), ce qui évite les chaînes A~B, B~C, C~D. Un article n'appartient qu'à un groupe ; une paire proche mais hors groupe a `groupe = 0`.
Les paires d'une **même marque** au-dessus du seuil sont listées à part (paires proches internes : pas forcément des doublons, et pas de la cannibalisation entre marques).
On ajoute le **recouvrement des paragraphes** : pour chaque paragraphe de A, la similarité avec son plus proche paragraphe de B (et inversement), en moyenne.
Proche de 1 = contenus presque identiques ; plus bas = même sujet, écrit autrement.

In [ ]:
etape3 = articles.drop_duplicates("url").set_index("url")
etape3["theme_txt"] = etape3["theme"].astype(str) + " · " + etape3["theme_nom"]
AXES4 = ["espece", "aliment_principal", "stade_principal", "sante"]

def recouvrement_paragraphes(a, b):
    "Moyenne, pour chaque paragraphe de A puis de B, de la similarité avec le paragraphe le plus proche de l'autre article."
    if a not in para_de or b not in para_de:
        return np.nan
    M = vecteurs_para[para_de[a]] @ vecteurs_para[para_de[b]].T
    return float((M.max(axis=1).mean() + M.max(axis=0).mean()) / 2)

def decrire(t):
    "Ajoute marque, titre, URL, thème, axes Mars et recouvrement des paragraphes aux deux articles de chaque paire."
    t = t.copy()
    for c in ["a", "b"]:
        r = art.loc[t[c]]
        t[f"marque_{c}"], t[f"titre_{c}"], t[f"url_{c}"] = r["marque"].values, r["titre"].values, r["url"].values
        e = etape3.reindex(r["url"].values)
        t[f"theme_{c}"] = e["theme_txt"].values
        for ax in AXES4:
            t[f"{ax}_{c}"] = e[ax].values
    t["meme_theme"] = t["theme_a"] == t["theme_b"]
    t["meme_case_mars"] = np.all([t[f"{ax}_a"] == t[f"{ax}_b"] for ax in AXES4], axis=0)
    t["recouvrement"] = [recouvrement_paragraphes(a, b) for a, b in zip(t["a"], t["b"])]
    return t

retenues = decrire(entre[entre["similarite"] >= SEUIL].sort_values("similarite", ascending=False))
meme_marque = decrire(paires[(~paires["entre_marques"]) & (paires["similarite"] >= SEUIL)].sort_values("similarite", ascending=False))
print("paires retenues", retenues.shape, "| paires proches dans une même marque", meme_marque.shape)
retenues[["similarite", "recouvrement", "marque_a", "titre_a", "marque_b", "titre_b"]].head(3)

arts = np.array(sorted(set(retenues["a"]) | set(retenues["b"])))
sim_min = float(retenues["similarite"].min())
D = np.clip(1 - S4[np.ix_(arts, arts)], 0, None)               # distance = 1 - similarité
np.fill_diagonal(D, 0)
lien = linkage(squareform(D, checks=False), method="complete")   # lien complet : tous proches de tous
etiquette = fcluster(lien, t=1 - sim_min, criterion="distance")
groupe_de = dict(zip(arts, etiquette))
retenues["groupe"] = [groupe_de[a] if groupe_de[a] == groupe_de[b] else 0 for a, b in zip(retenues["a"], retenues["b"])]
ordre = retenues[retenues["groupe"] > 0].groupby("groupe")["similarite"].max().sort_values(ascending=False)
numero = {g: i + 1 for i, g in enumerate(ordre.index)}         # groupe 1 = le plus similaire
numero[0] = 0                                                  # 0 = paire proche mais hors groupe complet
retenues["groupe"] = retenues["groupe"].map(numero)
matrice_marques = pd.DataFrame(0, index=marques, columns=marques)
for x, y in zip(retenues["marque_a"], retenues["marque_b"]):
    matrice_marques.loc[x, y] += 1
    matrice_marques.loc[y, x] += 1
print(f"{len(retenues)} paires de cannibalisation entre marques, {len(ordre)} groupes, {len(meme_marque)} paires proches dans une même marque")
matrice_marques

**Les groupes les plus nets** (pistes pour les cas de cannibalisation à présenter) : un groupe par ligne, avec ses articles et ses marques.

In [ ]:
def resume_groupes(retenues, n_max=8):
    lignes = []
    for g, sous in retenues[retenues["groupe"] > 0].groupby("groupe"):
        membres = sorted(set(sous["a"]) | set(sous["b"]))
        lignes.append({"groupe": g, "n_articles": len(membres), "similarite_max": round(sous["similarite"].max(), 3),
                       "marques": ", ".join(sorted(set(art.loc[membres, "marque"]))),
                       "titres": " | ".join(art.loc[membres, "titre"].str.slice(0, 45))})
    return pd.DataFrame(lignes).sort_values("groupe").head(n_max)

pd.set_option("display.max_colwidth", 130)
resume_groupes(retenues)

**Rapport lisible** (`rapport_cannibalisation.md`) : pour chaque groupe, la liste des articles (marque, titre, lien, thème, axes) et les paires avec leur similarité et leur recouvrement de paragraphes.

In [ ]:
def rapport_markdown(retenues):
    md = f"# Cannibalisation entre marques Mars\n\nSeuil : **similarité ≥ {SEUIL}**. Similarité médiane entre deux articles : {np.median(sim4):.3f}.\n\n"
    md += f"{len(retenues)} paires d'articles de marques différentes, dans {len(ordre)} groupes (dans un groupe, tous les articles sont proches deux à deux ; groupe 0 dans le CSV = paire hors groupe).\n\n---\n\n"
    for g, sous in retenues[retenues["groupe"] > 0].groupby("groupe"):
        membres = sorted(set(sous["a"]) | set(sous["b"]))
        md += f"## Groupe {g} ({len(membres)} articles) — similarité max {100 * sous['similarite'].max():.1f} %\n\n**Articles :**\n"
        for x in membres:
            r, e = art.loc[x], etape3.loc[art.at[x, "url"]]
            md += f"  - **{r['marque']}** : [{r['titre']}]({r['url']}) — thème {e['theme_txt']} · {e['espece']} / {e['aliment_principal']} / {e['stade_principal']}\n"
        md += "\n**Paires :**\n"
        for _, p in sous.iterrows():
            rec = "" if pd.isna(p["recouvrement"]) else f", paragraphes {100 * p['recouvrement']:.0f} %"
            md += f"  - {p['marque_a']} ↔ {p['marque_b']} : {100 * p['similarite']:.1f} %{rec}{' — même case Mars' if p['meme_case_mars'] else ''}\n"
        md += "\n---\n\n"
    return md

rapport = rapport_markdown(retenues)
print(len(rapport.splitlines()), "lignes de rapport")

## 16. Espaces vides
**Ce que ça fait** : compte les articles par case (animal × type d'aliment × stade de vie) et par marque, puis liste :
- les **cases vides pour toutes les marques** (« Mars-wide gap ») et les cases **peu couvertes** (≤ 2 articles) ;
- les **lacunes de marque** : une marque a 0 article dans une case où les autres marques en ont au moins `MIN_AUTRES_LACUNE` ;
- idem par **thème**.
**Règle de périmètre** : une marque ne « manque » une case que dans son propre périmètre (un animal et un type d'aliment sur lesquels elle publie vraiment, au moins `MIN_PERIMETRE` articles).
Ne pas publier de contenu chat (Cesar, Eukanuba, Pedigree) est un positionnement, pas une lacune. « dog and cat » compte pour les deux animaux ; « unspecified » est laissé de côté.

In [ ]:
FOODS = ["dry", "wet", "treats", "not food-specific"]
STAGES = ["puppy / kitten", "adult", "senior", "all ages"]
s3 = articles[articles["marque"].isin(marques)]               # 662 articles : les espagnols comptent ici
sp = s3.assign(animal=s3["espece"].map(lambda v: ["dog", "cat"] if v == "dog and cat" else [v])).explode("animal")
sp = sp[sp["animal"].isin(["dog", "cat"])]
g = sp.assign(aliment_1=sp["aliment"].str.split(" + ", regex=False)).explode("aliment_1")
g = g.assign(stade_1=g["stade"].str.split(" + ", regex=False)).explode("stade_1")
cases = pd.MultiIndex.from_product([["dog", "cat"], FOODS, STAGES], names=["animal", "aliment", "stade"])
grille_vides = (g.groupby(["animal", "aliment_1", "stade_1", "marque"]).size().unstack("marque", fill_value=0)
                .rename_axis(index=["animal", "aliment", "stade"]).reindex(cases, fill_value=0).reindex(columns=marques, fill_value=0))
grille_vides["TOTAL"] = grille_vides[marques].sum(axis=1)
themes_marques = (s3.assign(theme_txt=s3["theme"].astype(str) + " · " + s3["theme_nom"]).groupby(["theme_txt", "marque"]).size()
                  .unstack("marque", fill_value=0).reindex(columns=marques, fill_value=0))
themes_marques = themes_marques.loc[sorted(themes_marques.index, key=lambda v: int(v.split(" · ")[0]))]
themes_marques["TOTAL"] = themes_marques[marques].sum(axis=1)
print("grille", grille_vides.shape, "| thèmes x marques", themes_marques.shape)
grille_vides.head(8)

def perimetres(s3):
    """Pour chaque marque : les animaux et les types d'aliment sur lesquels elle publie (>= MIN_PERIMETRE articles)."""
    n_animal = s3[s3["espece"].isin(["dog", "cat"])].groupby(["marque", "espece"]).size()
    n_aliment = s3.assign(a=s3["aliment"].str.split(" + ", regex=False)).explode("a").groupby(["marque", "a"]).size()
    animaux = {b: {s for s in ["dog", "cat"] if n_animal.get((b, s), 0) >= MIN_PERIMETRE} for b in marques}
    aliments = {b: {f for f in ["dry", "wet", "treats"] if n_aliment.get((b, f), 0) >= MIN_PERIMETRE} | {"not food-specific"} for b in marques}
    return animaux, aliments

def lister_lacunes(grille, themes, animaux, aliments, animal_du_theme):
    lignes = []
    for (an, al, st), r in grille.iterrows():
        case = f"{an} · {al} · {st}"
        if r["TOTAL"] == 0:
            lignes.append({"type": "Mars-wide gap", "case": case, "marque": "all brands", "articles_dans_la_case": 0, "articles_autres_marques": 0})
        elif r["TOTAL"] <= 2:
            lignes.append({"type": "thin coverage", "case": case, "marque": "all brands", "articles_dans_la_case": int(r["TOTAL"]), "articles_autres_marques": int(r["TOTAL"])})
        for b in marques:
            autres = int(r["TOTAL"] - r[b])
            if r[b] == 0 and autres >= MIN_AUTRES_LACUNE and an in animaux[b] and al in aliments[b]:
                lignes.append({"type": "brand gap", "case": case, "marque": b, "articles_dans_la_case": 0, "articles_autres_marques": autres})
    for th, r in themes.iterrows():
        for b in marques:
            autres = int(r["TOTAL"] - r[b])
            dans_perimetre = animal_du_theme.get(th) not in ("dog", "cat") or animal_du_theme.get(th) in animaux[b]
            if r[b] == 0 and autres >= MIN_AUTRES_LACUNE and dans_perimetre:
                lignes.append({"type": "brand gap (theme)", "case": th, "marque": b, "articles_dans_la_case": 0, "articles_autres_marques": autres})
    return pd.DataFrame(lignes)

animaux, aliments = perimetres(s3)
animal_du_theme = s3.assign(theme_txt=s3["theme"].astype(str) + " · " + s3["theme_nom"]).groupby("theme_txt")["espece"].agg(lambda v: v.value_counts().index[0]).to_dict()
lacunes = lister_lacunes(grille_vides, themes_marques, animaux, aliments, animal_du_theme)
print("périmètre des marques :", {b: sorted(animaux[b]) for b in marques})
print(lacunes["type"].value_counts().to_dict())
lacunes.head(8)

**Enregistrement** des résultats de l'étape 4 dans le dossier de sorties.

In [ ]:
colonnes = ["groupe", "similarite", "recouvrement", "marque_a", "titre_a", "url_a", "theme_a", "marque_b", "titre_b", "url_b", "theme_b",
            "meme_theme", "meme_case_mars"] + [f"{ax}_{c}" for c in ["a", "b"] for ax in AXES4]
retenues[colonnes].round(3).to_csv(f"{OUT_DIR}/cannibalisation_paires.csv", index=False, encoding="utf-8-sig")
meme_marque[[c for c in colonnes if c != "groupe"]].round(3).to_csv(f"{OUT_DIR}/paires_proches_meme_marque.csv", index=False, encoding="utf-8-sig")
matrice_marques.to_csv(f"{OUT_DIR}/cannibalisation_matrice_marques.csv", encoding="utf-8-sig")
sensibilite.to_csv(f"{OUT_DIR}/seuil_sensibilite.csv", index=False, encoding="utf-8-sig")
grille_vides.to_csv(f"{OUT_DIR}/espaces_vides_grille.csv", encoding="utf-8-sig")
themes_marques.to_csv(f"{OUT_DIR}/espaces_vides_themes.csv", encoding="utf-8-sig")
lacunes.to_csv(f"{OUT_DIR}/espaces_vides_lacunes.csv", index=False, encoding="utf-8-sig")
with open(f"{OUT_DIR}/rapport_cannibalisation.md", "w", encoding="utf-8") as f:
    f.write(rapport)
print(sorted(os.listdir(OUT_DIR)))

## 17. Données du tableau de bord
**Ce que ça fait** : prépare les données de la page unique « tableau de bord » (vue d'ensemble, carte, cannibalisation, espaces vides, seuil) : articles, paires pour le **curseur de seuil** (toutes les paires ≥ 0,75, pour que l'utilisateur
puisse déplacer le seuil entre `MIN_CURSEUR` et `MAX_CURSEUR`, 0,75 et 0,95 par défaut), grille et lacunes de l'étape 4. La page elle-même (HTML et JavaScript) et le zip de livraison sont dans l'**Annexe**, tout en bas.
**Paramètre qui compte** : `SEUIL`, qui devient la valeur par défaut du curseur (le curseur va de `MIN_CURSEUR` à `MAX_CURSEUR`). **Pourquoi pas 0 ?** La similarité médiane entre deux articles est de 0,575 : presque tous se ressemblent un peu. À 0,65 il y aurait plus de 20 000 paires entre marques (au lieu de 1 547 à 0,75) : la page deviendrait lourde, la carte illisible, et ces paires ne sont que des sujets voisins, pas des articles qui se concurrencent. On peut descendre jusqu'à `MIN_CANDIDAT` (0,70 par défaut).

In [ ]:
assert MIN_CANDIDAT <= MIN_CURSEUR <= SEUIL <= MAX_CURSEUR, "il faut MIN_CANDIDAT <= MIN_CURSEUR <= SEUIL <= MAX_CURSEUR"
curseur = paires[paires["similarite"] >= MIN_CURSEUR]               # paires proposées au curseur (658 articles de l'étape 2)
liens = [[int(a), int(b), round(float(s), 5), None if np.isnan(r) else round(r, 3), int(e)]
         for (a, b, s, e), r in zip(curseur[["a", "b", "similarite", "entre_marques"]].itertuples(index=False),
                                    [recouvrement_paragraphes(a, b) for a, b in zip(curseur["a"], curseur["b"])])]
print(f"{len(liens)} paires >= {MIN_CURSEUR} ({sum(p[4] for p in liens)} entre marques)")

def cases_mars(r):
    "Cases de la grille Mars de l'article : animal x aliment x stade (un article à plusieurs étiquettes compte dans chacune)."
    animaux_r = ["dog", "cat"] if r["espece"] == "dog and cat" else [r["espece"]]
    return [f"{an} · {al} · {st}" for an in animaux_r for al in r["aliment"].split(" + ") for st in r["stade"].split(" + ")]

noms_page = {"article_id": "id", "titre": "title", "marque": "brand", "langue": "language", "theme_nom": "theme_name",
             "famille_theme": "family", "famille_article": "article_family", "espece": "pet_type", "aliment": "food_type",
             "stade": "life_stage", "sante": "health_need", "sante_sous_theme": "health_subtheme", "rang_dans_theme": "rank_in_theme"}
articles_page = articles.assign(cells=articles.apply(cases_mars, axis=1), x=articles["umap_x"].round(3), y=articles["umap_y"].round(3))
articles_page = articles_page.rename(columns=noms_page)[["id", "title", "brand", "url", "language", "theme", "theme_name", "family",
    "article_family", "pet_type", "food_type", "life_stage", "health_need", "health_subtheme", "rank_in_theme", "cells", "x", "y"]]
articles_page.shape

grille_page = grille_vides.reset_index().rename(columns={"animal": "species", "aliment": "food_type", "stade": "life_stage"})
lacunes_page = lacunes.rename(columns={"case": "cell", "marque": "brand", "articles_dans_la_case": "articles_in_cell",
                                       "articles_autres_marques": "sister_brand_articles"})
themes_page = resume_themes.rename(columns={"nom": "name", "famille": "family", "titre_representatif": "representative_title"})
donnees_page = {"articles": articles_page.to_dict(orient="records"), "pairs": liens, "annot": [], "t0": SEUIL, "tmin": MIN_CURSEUR, "tmax": MAX_CURSEUR,
                "grid": grille_page.to_dict(orient="records"), "brands": marques,
                "scope": {b: [an for an in ("dog", "cat") if an in animaux[b]] for b in marques},   # animaux sur lesquels chaque marque publie
                "gaps": lacunes_page.to_dict(orient="records"), "names": {},
                "themes": themes_page[["theme", "name", "family", "n_articles", "representative_title"]].to_dict(orient="records")}
print({k: (len(v) if hasattr(v, "__len__") else v) for k, v in donnees_page.items()})

---
# Annexe — affichage : la carte interactive
**Cette section n'est pas du calcul** : elle fabrique une page HTML autonome (carte Plotly + fiche d'article au clic + recherche) à partir du tableau `articles`.
Elle est volontairement peu détaillée : le HTML, le CSS et le JavaScript sont dans des chaînes de texte. Pour comprendre les résultats, la section 1 à 12 suffit.

**Couleurs et zones.** Une zone colorée entoure les articles centraux d'un thème (enveloppe convexe) ; pas de zone si le thème est trop dispersé (`ZONE_MAX`).

In [ ]:
COULEURS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
            "#393b79", "#ad494a", "#637939", "#8c6d31", "#843c39", "#7b4173", "#3182bd", "#e6550d", "#31a354", "#756bb1"]
COULEURS_FIXES = {"dog": "#d97706", "cat": "#2563eb", "dog and cat": "#0f9d76", "unspecified": "#9ca3af", "dry": "#b45309",
                  "wet": "#0ea5e9", "treats": "#16a34a", "not food-specific": "#cbd5e1", "puppy / kitten": "#f59e0b",
                  "adult": "#2563eb", "senior": "#7c3aed", "all ages": "#cbd5e1", "yes": "#dc2626", "no": "#cbd5e1", "": "#e2e8f0"}

def surface(xy):
    return float(np.ptp(xy[:, 0]) * np.ptp(xy[:, 1])) or None        # aire du rectangle qui contient la carte

def zone_theme(xy, aire):
    """Contour des PART_ZONE articles les plus proches du centre du thème (None si trop petit ou dispersé)."""
    if len(xy) < 6:
        return None
    d = np.linalg.norm(xy - np.median(xy, axis=0), axis=1)
    garde = xy[d <= np.quantile(d, PART_ZONE)]
    try:
        hull = ConvexHull(garde)
    except Exception:
        return None                                                    # points alignés : pas de zone
    if aire and hull.volume > ZONE_MAX * aire:                         # en 2D, volume = aire
        return None
    pts = garde[hull.vertices]
    return np.vstack([pts, pts[:1]])

**Traces de la carte** : les zones (sous les points), puis un groupe de points par mode de couleur (thème, famille, marque, langue, animal, aliment, stade, santé).

In [ ]:
MODES = [("theme", "theme", "Couleur : thème"), ("famille", "famille_theme", "Couleur : famille Mars (thème)"),
         ("famille_article", "famille_article", "Couleur : famille Mars (article)"), ("marque", "marque", "Couleur : marque"),
         ("langue", "langue", "Couleur : langue"), ("espece", "espece", "Animal (chien / chat)"),
         ("aliment", "aliment_principal", "Type d'aliment"), ("stade", "stade_principal", "Stade de vie"),
         ("sante", "sante", "Besoin santé (oui / non)"), ("sante_sous_theme", "sante_sous_theme", "Besoin santé (sous-thèmes)")]

def traces_zones(a):
    traces, aire = [], surface(a[["umap_x", "umap_y"]].to_numpy())
    for c in sorted(a["theme"].unique()):
        z = zone_theme(a.loc[a["theme"] == c, ["umap_x", "umap_y"]].to_numpy(), aire)
        if z is not None:
            couleur = COULEURS[c % len(COULEURS)]
            traces.append({"type": "scatter", "x": z[:, 0].round(4).tolist(), "y": z[:, 1].round(4).tolist(), "mode": "lines",
                           "fill": "toself", "fillcolor": couleur, "opacity": 0.13, "line": {"color": couleur, "width": 1},
                           "hoverinfo": "skip", "showlegend": False})
    return traces

def libelle(n_groupe, total):
    return f"{n_groupe} ({100 * n_groupe / total:.1f}%)"                # effectif et part, affichés dans la légende

def couleur_groupe(mode, g, i, d):
    if mode == "theme":
        return COULEURS[int(g) % len(COULEURS)]
    if mode.startswith("famille") and g in FAMILLES:                    # même couleur pour une famille dans les 2 modes
        return COULEURS[(list(FAMILLES).index(g) * 3) % len(COULEURS)]
    if g in COULEURS_FIXES and mode != "sante_sous_theme" or (mode == "sante_sous_theme" and g == ""):
        return COULEURS_FIXES[g]
    return COULEURS[(i * 3) % len(COULEURS)]

def texte_survol(d):
    return (d["titre"].str.slice(0, 90) + "<br>" + d["marque"] + " · " + d["espece"] + " · " + d["aliment"] + " · " + d["stade"]
            + " · santé " + d["sante"] + "<br>thème " + d["theme"].astype(str) + " · " + d["theme_nom"]).tolist()

def ajouter_points(a, traces):
    """Ajoute un groupe de traces par mode ; renvoie {mode: indices des traces}."""
    visibles = {m: [] for m, _, _ in MODES}
    for mode, col, _ in MODES:
        groupes = sorted(a[col].unique()) if mode == "theme" else sorted(a[col].unique(), key=lambda v: (str(v) == "", str(v)))
        for i, g in enumerate(groupes):
            d = a[a[col] == g]
            nom = f"{g} · {d['theme_nom'].iloc[0]}" if mode == "theme" else (str(g) or "pas un besoin santé")
            traces.append({"type": "scatter", "mode": "markers", "x": d["umap_x"].tolist(), "y": d["umap_y"].tolist(),
                           "name": f"{nom[:55]} — {libelle(len(d), len(a))}", "visible": mode == "theme", "legendgroup": mode,
                           "marker": {"size": 9, "opacity": 0.85, "color": couleur_groupe(mode, g, i, d), "line": {"width": 0.5, "color": "white"}},
                           "customdata": d["article_id"].tolist(), "hovertemplate": "%{text}<extra></extra>", "text": texte_survol(d)})
            visibles[mode].append(len(traces) - 1)
    return visibles

**Figure complète** : deux vues (« notre classification » et « axes Mars »), menus de couleur et de zones, étiquettes des thèmes, trace de surlignage pour la recherche.

In [ ]:
def annotations_themes(a):
    notes = [{"x": float(g["umap_x"].median()), "y": float(g["umap_y"].median()), "showarrow": False,
              "text": f"<b>{c}</b> {g['theme_nom'].iloc[0][:28]}", "font": {"size": 11, "color": COULEURS[int(c) % len(COULEURS)]},
              "bgcolor": "rgba(255,255,255,0.75)", "borderpad": 2} for c, g in a.groupby("theme")]
    notes += [{"x": float(g["umap_x"].mean()), "y": float(g["umap_y"].max()), "showarrow": False, "yshift": 14,
               "text": f"<b>{html.escape(f.upper())}</b>", "font": {"size": 13, "color": "#1f2933"}, "opacity": 0.75}
              for f, g in a.groupby("famille_theme")]
    return notes

def menu_couleurs(modes, visibles, points):
    boutons = [{"label": lab, "method": "restyle", "args": [{"visible": [i in visibles[m] for i in points]}, points]}
               for m, _, lab in MODES if m in modes]
    return {"buttons": boutons, "direction": "down", "x": 0, "y": 1.1, "xanchor": "left"}

def menu_zones(zones):
    return {"buttons": [{"label": "Zones : oui", "method": "restyle", "args": [{"visible": True}, zones]},
                        {"label": "Zones : non", "method": "restyle", "args": [{"visible": False}, zones]}],
            "direction": "down", "x": 0.25, "y": 1.1, "xanchor": "left"}

def layout_carte(titre, vues):
    axe = {"showgrid": False, "zeroline": False}
    return {"title": {"text": titre}, "height": 760, "template": "plotly_white", "hovermode": "closest", "dragmode": "pan",
            "clickmode": "event", "xaxis": {"title": "UMAP 1", **axe}, "yaxis": {"title": "UMAP 2", **axe},
            "legend": {"title": {"text": "Articles (part du total) · cliquer pour masquer"}, "font": {"size": 10}},
            "annotations": vues["themes"]["annotations"], "updatemenus": vues["themes"]["updatemenus"], "meta": {"vues": vues}}

def figure_carte(a, titre):
    traces = traces_zones(a)
    zones = list(range(len(traces)))
    visibles = ajouter_points(a, traces)
    points = sum(visibles.values(), [])
    traces.append({"type": "scatter", "mode": "markers", "x": [], "y": [], "name": "recherche", "hoverinfo": "skip",
                   "showlegend": False, "marker": {"size": 17, "symbol": "circle-open", "color": "black", "line": {"width": 2.5}}})
    def visible(mode, avec_zones):
        return [(i in zones and avec_zones) or i in visibles[mode] or i == len(traces) - 1 for i in range(len(traces))]
    vues = {"themes": {"updatemenus": [menu_couleurs(["theme", "famille", "famille_article", "marque", "langue"], visibles, points),
                                       menu_zones(zones)],
                       "annotations": annotations_themes(a), "visible": visible("theme", True)},
            "mars": {"updatemenus": [menu_couleurs(["espece", "aliment", "stade", "sante", "sante_sous_theme", "marque"], visibles, points)],
                     "annotations": [], "visible": visible("espece", False)}}
    for i, v in enumerate(vues["themes"]["visible"]):
        traces[i]["visible"] = v
    return {"data": traces, "layout": layout_carte(titre, vues)}

**Style de la page** (CSS).

In [ ]:
STYLE = """<style>
body { font-family: Arial, sans-serif; margin: 24px; color: #222; background: #fff; }
.aide { color: #555; }
.panneau { display: grid; grid-template-columns: minmax(280px, 1fr) minmax(280px, 1.3fr); gap: 16px; margin: 6px 0 40px; }
@media (max-width: 900px) { .panneau { grid-template-columns: 1fr; } }
.fiche, .liste { border: 1px solid #ccc; border-radius: 8px; padding: 12px 16px; background: #f7f7f7; line-height: 1.6; }
.fiche h3, .liste h3 { margin: 0 0 8px; }
.fiche table { border-collapse: collapse; width: 100%; }
.fiche td { padding: 3px 6px; vertical-align: top; border-bottom: 1px solid #e3e3e3; overflow-wrap: anywhere; }
.fiche td:first-child { color: #555; white-space: nowrap; width: 1%; }
.liste ol { max-height: 360px; overflow-y: auto; margin: 0; padding-left: 22px; }
.liste li { margin: 3px 0; cursor: pointer; }
.liste li:hover { text-decoration: underline; }
.liste li.actif { font-weight: bold; }
.pastille { display: inline-block; width: 12px; height: 12px; border-radius: 3px; margin-right: 6px; vertical-align: -1px; }
a { color: #155eef; }
.recherche { margin: 10px 0 4px; }
.recherche input { width: min(560px, 90%); padding: 7px 10px; font-size: 14px; border: 1px solid #bbb; border-radius: 6px; }
.vues { margin: 8px 0 2px; }
.vue { font-size: 14px; padding: 6px 12px; margin-right: 6px; border: 1px solid #bbb; border-radius: 6px; background: #fff; cursor: pointer; }
.vue.actif { background: #1f2933; color: #fff; border-color: #1f2933; }
</style>"""

**JavaScript (1/4)** : petits outils (échappement du texte, ligne de tableau, hauteur du cadre, changement de vue).

In [ ]:
JS_OUTILS = r"""<script>
function esc(s) { return String(s ?? "").replace(/[&<>"']/g, m => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[m])); }
function ligne(nom, valeur) {
  if (valeur === null || valeur === undefined || valeur === "") return "";
  return "<tr><td>" + esc(nom) + "</td><td>" + valeur + "</td></tr>";
}
function ajuster() {
  try { const cadre = window.frameElement; if (cadre) cadre.style.height = (document.documentElement.scrollHeight + 20) + "px"; } catch (e) {}
}
function vue(divId, nom) {
  const gd = document.getElementById(divId);
  if (!gd || !gd.layout || !gd.layout.meta) return;
  const V = gd.layout.meta.vues[nom];
  Plotly.restyle(gd, {visible: V.visible});
  Plotly.relayout(gd, {updatemenus: V.updatemenus, annotations: V.annotations});
  ["themes", "mars"].forEach(n => document.getElementById(divId + "-vue-" + n).classList.toggle("actif", n === nom));
}
</script>"""

**JavaScript (2/4)** : fiche d'un article et liste des articles du même thème, affichées au clic sur un point.

In [ ]:
JS_FICHE = r"""<script>
function afficher(divId, id) {
  const A = ARTICLES[id];
  if (!A) return;
  const couleur = COULEURS_THEMES[A.theme] || "#999";
  document.getElementById(divId + "-fiche").innerHTML = "<h3>Article n°" + esc(A.article_id) + "</h3><table>" +
    ligne("Titre", "<b>" + esc(A.titre) + "</b>") + ligne("Marque", esc(A.marque)) + ligne("Langue", esc(A.langue)) +
    ligne("Lien", '<a href="' + esc(A.url) + '" target="_blank" rel="noopener">' + esc(A.url) + "</a>") +
    ligne("Thème", '<span class="pastille" style="background:' + couleur + '"></span>' + esc(A.theme) + " · " + esc(A.theme_nom)) +
    ligne("Famille Mars (thème)", esc(A.famille_theme)) + ligne("Famille Mars (article)", esc(A.famille_article)) +
    ligne("Animal", "<b>" + esc(A.espece) + "</b>") + ligne("Animal (thème)", esc(A.espece_theme)) +
    ligne("Type d'aliment", "<b>" + esc(A.aliment) + "</b>") + ligne("Stade de vie", "<b>" + esc(A.stade) + "</b>") +
    ligne("Besoin santé", "<b>" + esc(A.sante) + "</b>" + (A.sante_sous_theme ? " · " + esc(A.sante_sous_theme) : "")) +
    ligne("Source des axes", "<span class='aide'>" + esc(A.source_axes) + "</span>") +
    ligne("Rang dans le thème", esc(A.rang_dans_theme) + " / " + esc(TAILLES[A.theme]) + " (1 = le plus central)") +
    ligne("Silhouette", Number(A.silhouette).toFixed(2) + " (thème voisin : " + esc(A.voisin_nom) + ")") +
    ligne("Mots", esc(A.n_mots)) + ligne("Description", esc(A.description)) + "</table>";
  const membres = ARTICLES.filter(x => x.theme === A.theme).sort((a, b) => a.rang_dans_theme - b.rang_dans_theme);
  let liste = "<h3>Thème " + esc(A.theme) + " · " + esc(A.theme_nom) + " — " + membres.length + " articles (" +
    (100 * membres.length / ARTICLES.length).toFixed(1) + "%)</h3><p class='aide'>Du plus central au plus éloigné. " +
    "Cliquer sur un titre pour voir sa fiche.</p><ol>";
  membres.forEach(m => {
    liste += '<li class="' + (m.article_id === A.article_id ? "actif" : "") + '" onclick="afficher(\'' + divId + "', " +
      m.article_id + ')">' + esc(m.titre) + " <span class='aide'>(" + esc(m.marque) + ")</span></li>";
  });
  document.getElementById(divId + "-liste").innerHTML = liste + "</ol>";
  ajuster();
}
</script>"""

**JavaScript (3/4)** : recherche par mots dans les titres.

In [ ]:
JS_RECHERCHE = r"""<script>
function rechercher(divId) {
  const texte = document.getElementById(divId + "-recherche").value.trim().toLowerCase();
  const termes = texte.split(/[\s,;]+/).filter(m => m.length >= 2);
  const trouves = termes.length ? ARTICLES.filter(a => {
    const t = (a.titre + " " + a.slug + " " + (a.description || "")).toLowerCase();
    return termes.some(m => t.includes(m));
  }) : [];
  const gd = document.getElementById(divId);
  if (gd && gd.data) Plotly.restyle(gd, {x: [trouves.map(a => a.umap_x)], y: [trouves.map(a => a.umap_y)]}, [gd.data.length - 1]);
  const info = document.getElementById(divId + "-resultat");
  if (!termes.length) { info.textContent = ""; return; }
  info.textContent = trouves.length + " articles";
  let liste = "<h3>Recherche « " + esc(texte) + " » — " + trouves.length + " articles</h3>" +
    "<p class='aide'>Entourés en noir sur la carte. Cliquer sur un titre pour voir sa fiche.</p><ol>";
  trouves.forEach(m => {
    liste += '<li onclick="afficher(\'' + divId + "', " + m.article_id + ')">' + esc(m.titre) +
      " <span class='aide'>(" + esc(m.marque) + " · thème " + esc(m.theme) + ")</span></li>";
  });
  document.getElementById(divId + "-liste").innerHTML = liste + "</ol>";
  ajuster();
}
</script>"""

**JavaScript (4/4)** : tracé de la carte (attend que plotly.js soit chargé) et clic sur un point.

In [ ]:
JS_DESSIN = r"""<script>
function dessiner(divId, figure, essai) {
  if (typeof Plotly === "undefined") {
    if ((essai || 0) < 100) setTimeout(() => dessiner(divId, figure, (essai || 0) + 1), 100);
    else document.getElementById(divId).innerHTML = "<p class='aide'>plotly.js n'a pas pu être chargé : ouvrir le fichier HTML dans un navigateur.</p>";
    return;
  }
  Plotly.newPlot(divId, figure.data, figure.layout, {scrollZoom: true, responsive: true, displaylogo: false})
    .then(gd => gd.on("plotly_click", ev => {
      const p = ev.points && ev.points[0];
      if (p && p.customdata !== undefined && p.customdata !== null) afficher(divId, p.customdata);
    })).then(ajuster);
}
</script>"""

**Assemblage de la page** et enregistrement dans `outputs/carte_interactive.html` (autonome : plotly.js est intégré, ouverture hors connexion possible).
Dans le notebook on affiche la même page avec plotly.js chargé depuis Internet (cela garde le notebook léger).

In [ ]:
COLONNES_PAGE = ["article_id", "slug", "marque", "langue", "titre", "url", "theme", "theme_nom", "famille_theme", "famille_article",
                 "espece", "espece_theme", "aliment", "stade", "sante", "sante_sous_theme", "source_axes", "silhouette",
                 "voisin_nom", "rang_dans_theme", "n_mots", "description", "umap_x", "umap_y"]

def construire_page(a, plotly_integre=True):
    def js(objet):                                              # JSON sûr dans une balise <script>
        return json.dumps(objet, ensure_ascii=False, default=str).replace("</", "<\\/")
    donnees = a.sort_values("article_id")[COLONNES_PAGE].replace({np.nan: None}).to_dict(orient="records")
    couleurs = {int(c): COULEURS[int(c) % len(COULEURS)] for c in a["theme"].unique()}
    tailles = {int(c): int(v) for c, v in a["theme"].value_counts().items()}
    script_donnees = f"<script>const ARTICLES = {js(donnees)};\nconst COULEURS_THEMES = {js(couleurs)};\nconst TAILLES = {js(tailles)};</script>"
    script_plotly = ("<script>" + get_plotlyjs() + "</script>") if plotly_integre else "<script src='https://cdn.plot.ly/plotly-2.35.2.min.js'></script>"
    d, titre = "carte_umap", f"Carte UMAP — fidélité {fidelite}"
    corps = (f"<h1>Carte sémantique — {a['theme'].nunique()} thèmes, {len(a)} articles, {a['marque'].nunique()} marques</h1>"
             "<p class='aide'>Molette : zoom · glisser : déplacer · double-clic : vue complète · survol : titre · clic : fiche de l'article "
             "et articles du même thème · menus : couleurs et zones · boutons « Vue » : classification ou axes Mars. "
             "Les axes d'UMAP n'ont pas de signification : une zone vide n'est pas une lacune de contenu.</p>"
             f"<div class='recherche'><input id='{d}-recherche' type='search' placeholder='Chercher dans les titres, ex. groom, coat, bath' "
             f"oninput=\"rechercher('{d}')\"> <span id='{d}-resultat' class='aide'></span></div>"
             f"<div class='vues'><span class='aide'>Vue :</span> <button id='{d}-vue-themes' class='vue actif' onclick=\"vue('{d}','themes')\">"
             f"Notre classification (thèmes)</button><button id='{d}-vue-mars' class='vue' onclick=\"vue('{d}','mars')\">"
             f"Axes Mars (animal · aliment · stade · santé)</button></div><div id='{d}'></div>"
             f"<div class='panneau'><div class='fiche' id='{d}-fiche'>Cliquer sur un point pour voir la fiche de l'article.</div>"
             f"<div class='liste' id='{d}-liste'></div></div>"
             f"<script>dessiner('{d}', {js(figure_carte(a, titre))});</script>")
    return ("<!DOCTYPE html><html lang='fr'><head><meta charset='utf-8'><title>Carte sémantique Mars</title>" + script_plotly
            + STYLE + script_donnees + JS_OUTILS + JS_FICHE + JS_RECHERCHE + JS_DESSIN + "</head><body>" + corps + "</body></html>")

with open(f"{OUT_DIR}/carte_interactive.html", "w", encoding="utf-8") as f:
    f.write(construire_page(articles, plotly_integre=True))
print("Carte enregistrée :", f"{OUT_DIR}/carte_interactive.html", f"({os.path.getsize(f'{OUT_DIR}/carte_interactive.html') / 1e6:.1f} Mo)")
try:
    from IPython.display import HTML, display
    page = construire_page(articles, plotly_integre=False)
    display(HTML('<iframe srcdoc="' + html.escape(page, quote=True) + '" style="width:100%; height:2600px; border:0; background:#fff"></iframe>'))
except Exception as e:
    print("Affichage dans le notebook impossible :", e)

---
# Annexe 2 — le tableau de bord (page HTML) et le zip de livraison
**Cette section n'est pas du calcul** : elle contient le gabarit de la page (CSS, HTML, JavaScript, en chaînes de texte) et l'assemble avec les données de la section 17.
La page n'a besoin d'aucune bibliothèque : ses graphiques sont dessinés en SVG, elle s'ouvre donc hors connexion. Elle est disponible en **français et en anglais** (boutons FR / EN en haut à droite, français par défaut) : la traduction est faite à l'affichage à partir d'un dictionnaire ; les valeurs des données (noms de thèmes, titres, étiquettes `dog`, `dry`…) restent en anglais.
Onglets : Overview · Semantic map · Cannibalization · White space · Threshold ; un curseur règle le seuil de cannibalisation pour toutes les vues.

**Gabarit de la page (1/11)** : style (CSS).

In [ ]:
PAGE_1 = r"""
<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Mars content map</title>
<style>
:root{--ink:#1f2933;--mute:#5f6b7a;--line:#dfe3e8;--bg:#f6f7f9;--card:#fff;--acc:#2563eb;--gap:#d03b3b;--gapbg:#fdecec;--ok:#0f9d76;--warn:#d97706}
*{box-sizing:border-box}
body{margin:0;font-family:Inter,"Segoe UI",Arial,sans-serif;color:var(--ink);background:var(--bg);font-size:14px;line-height:1.45}
header{background:var(--ink);color:#fff;padding:16px 24px 0}
header h1{margin:0;font-size:22px;font-weight:700}
header p{margin:4px 0 12px;color:#c9d2dc;font-size:13px}
nav{display:flex;gap:4px;flex-wrap:wrap}
nav button{background:transparent;color:#c9d2dc;border:0;padding:10px 16px;font-size:14px;font-weight:600;cursor:pointer;border-radius:8px 8px 0 0}
nav button.on{background:var(--bg);color:var(--ink)}
.bar{position:sticky;top:0;z-index:20;background:#fff;border-bottom:1px solid var(--line);padding:10px 24px;display:flex;gap:22px;align-items:center;flex-wrap:wrap}
.bar label{font-weight:700}
.bar input[type=range]{width:260px;vertical-align:middle}
.tval{font-family:Consolas,monospace;font-weight:700;font-size:16px;color:var(--acc);min-width:52px;display:inline-block}
.kpi{display:flex;gap:18px;flex-wrap:wrap}
.kpi div{font-size:12px;color:var(--mute)} .kpi b{display:block;font-size:18px;color:var(--ink)}
main{padding:18px 24px 40px}
section{display:none} section.on{display:block}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:14px 16px;margin-bottom:14px}
.card h2{margin:0 0 8px;font-size:16px}
.mute{color:var(--mute);font-size:13px}
.grid2{display:grid;grid-template-columns:minmax(0,1.6fr) minmax(320px,1fr);gap:14px}
.grid3{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}
@media (max-width:1000px){.grid2,.grid3{grid-template-columns:1fr}}
.big{font-size:30px;font-weight:700} .hl{color:var(--acc)}
#map{width:100%;height:640px;background:#fff;border:1px solid var(--line);border-radius:12px;cursor:grab;touch-action:none;display:block}
"""

**Gabarit de la page (2/11)** : style (CSS).

In [ ]:
PAGE_2 = r"""
.tools{display:flex;gap:10px;flex-wrap:wrap;align-items:center;margin-bottom:8px}
select,input[type=search]{font-size:13px;padding:6px 8px;border:1px solid #c8ced6;border-radius:8px;background:#fff}
input[type=search]{min-width:240px}
.legend{display:flex;flex-wrap:wrap;gap:4px 12px;margin-top:8px;font-size:12px}
.legend span{display:inline-flex;align-items:center;gap:5px;cursor:pointer;max-width:330px}
.legend span.off{opacity:.3}
.legend i{width:11px;height:11px;border-radius:3px;flex:0 0 11px}
.legend em{font-style:normal;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#panel{max-height:700px;overflow:auto;position:sticky;top:70px}
table{border-collapse:collapse;width:100%}
td,th{padding:5px 7px;border-bottom:1px solid #eef0f3;text-align:left;vertical-align:top;overflow-wrap:anywhere}
th{font-size:12px;color:var(--mute);font-weight:700;background:#fafbfc;position:sticky;top:0}
td.nw,th.nw{white-space:nowrap;overflow-wrap:normal}
td.n,th.n{text-align:right;font-family:Consolas,monospace;white-space:nowrap}
.kv td:first-child{color:var(--mute);white-space:nowrap;width:1%}
.tag{display:inline-block;padding:1px 8px;border-radius:99px;font-size:12px;font-weight:600;background:#eef2f7;margin:1px 2px 1px 0}
.tag.red{background:var(--gapbg);color:var(--gap)} .tag.green{background:#e6f6f0;color:var(--ok)}
a{color:var(--acc)} .link{color:var(--acc);cursor:pointer;text-decoration:underline}
.scroll{max-height:520px;overflow:auto;border:1px solid var(--line);border-radius:8px}
.heat td.c{text-align:center;font-weight:700;cursor:pointer;min-width:62px;font-family:Consolas,monospace}
.heat td.c:hover{outline:2px solid var(--ink)}
.heat td.sel{outline:3px solid var(--ink)}
.tip{position:fixed;pointer-events:none;background:var(--ink);color:#fff;padding:6px 9px;border-radius:6px;font-size:12px;max-width:340px;z-index:50;display:none}
.note{border-left:4px solid var(--warn);background:#fff8ec;padding:8px 12px;border-radius:6px;margin:8px 0;font-size:13px}
details summary{cursor:pointer;font-weight:600;margin:6px 0}
.chart{width:100%;height:340px}
button.mode{font-size:13px;font-weight:600;padding:7px 14px;border:1px solid #c8ced6;border-radius:8px;background:#fff;cursor:pointer}
button.mode.on{background:var(--ink);color:#fff;border-color:var(--ink)}
"""

**Gabarit de la page (3/11)**.

In [ ]:
PAGE_3 = r"""
header{position:relative}#langs{position:absolute;top:14px;right:24px;display:flex;gap:4px}#langs button{background:transparent;color:#c9d2dc;border:1px solid #5f6b7a;border-radius:6px;padding:4px 10px;font-weight:700;cursor:pointer}#langs button.on{background:#fff;color:var(--ink)}
</style></head><body>
<header><h1>Mars Petcare — content map, cannibalization and white space</h1>
<p>5 brands · articles classified by meaning (bge-m3) · themes (K-means) and the 4 Mars axes · one threshold drives every view</p>
<nav id="nav"></nav><div id="langs"><button data-l="fr">FR</button><button data-l="en">EN</button></div></header>
<div class="bar"><span><label>Cannibalization threshold</label> &nbsp;<input id="thr" type="range" min="0.75" max="0.95" step="0.001">
 <span class="tval" id="tval"></span> <span class="link" id="reset">reset to default</span></span>
 <div class="kpi" id="kpi"></div></div>
<main>
<section id="s-overview"></section>
<section id="s-map"><div class="grid2"><div>
  <div class="tools"><label>Colour</label><select id="colour"></select>
  <label><input type="checkbox" id="links" checked> cannibalization links</label>
  <label><input type="checkbox" id="labels" checked> labels</label>
  <input type="search" id="search" placeholder="Search titles (e.g. dental, senior cat, wet food)"><span class="mute" id="found"></span>
  <span class="link" id="zoomreset">reset zoom</span></div>
  <svg id="map"></svg><div class="legend" id="legend"></div>
  <p class="mute">Wheel: zoom · drag: pan · click a point: article card. Red lines join articles of different brands with similarity ≥ threshold. UMAP axes have no meaning; only distances count.</p>
</div><div class="card" id="panel"><h2>Article card</h2><p class="mute">Click a point on the map, or a title anywhere in the page.</p></div></div></section>
<section id="s-cann"></section>
<section id="s-white"></section>
<section id="s-thr"></section>
</main><div class="tip" id="tip"></div>
<script>
const D = __DATA__;
const A = D.articles, P = D.pairs, $ = s => document.querySelector(s);
const esc = s => String(s ?? "").replace(/[&<>"']/g, m => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[m]));
const PAL = ["#2563eb","#e8590c","#0f9d76","#c2255c","#7048e8","#d9a400","#1098ad","#5c940d","#e64980","#495057","#f76707","#4263eb","#37b24d","#ae3ec9","#a61e4d","#1c7ed6","#f59f00","#0ca678"];
"""

**Gabarit de la page (4/11)**.

In [ ]:
PAGE_4 = r"""
const FIX = {"dog":"#d97706","cat":"#2563eb","dog and cat":"#0f9d76","unspecified":"#adb5bd","dry":"#b45309","wet":"#0ea5e9","treats":"#16a34a",
  "not food-specific":"#ced4da","puppy / kitten":"#f59f00","adult":"#2563eb","senior":"#7c3aed","all ages":"#ced4da","yes":"#d03b3b","no":"#ced4da"};
let T = D.t0, sel = null, hidden = new Set(), matrixSel = null;
const pct = (v, d) => v == null ? "–" : (100 * v).toFixed(d) + " %";
const thr = $("#thr"); thr.min = D.tmin; thr.max = D.tmax; thr.value = T;
// ---------- tabs
const TABS = [["overview","Overview"],["map","Semantic map"],["cann","Cannibalization"],["white","White space"],["thr","Threshold"]];
$("#nav").innerHTML = TABS.map(([k,l]) => `<button data-k="${k}">${l}</button>`).join("");
function tab(k){ document.querySelectorAll("nav button").forEach(b => b.classList.toggle("on", b.dataset.k===k));
  document.querySelectorAll("section").forEach(s => s.classList.toggle("on", s.id==="s-"+k)); if (k==="map") draw(); if (k==="thr") drawThr(); fit(); }
document.querySelectorAll("nav button").forEach(b => b.onclick = () => tab(b.dataset.k));
// ---------- pairs at the current threshold
function cross(){ return P.filter(p => p[4] && p[2] >= T); }
function same(){ return P.filter(p => !p[4] && p[2] >= T); }
function kpi(){
  const c = cross(), arts = new Set(); c.forEach(([a,b]) => { arts.add(a); arts.add(b); });
  $("#tval").textContent = T.toFixed(3);
  $("#kpi").innerHTML = `<div><b>${c.length}</b>cross-brand pairs</div><div><b>${arts.size}</b>articles involved</div><div><b>${same().length}</b>same-brand similar pairs</div>`;
}
// ---------- overview
function overview(){
  const brands = D.brands, nb = {}; A.forEach(a => nb[a.brand] = (nb[a.brand]||0)+1);
  const mw = D.gaps.filter(g => g.type==="Mars-wide gap"), bg = D.gaps.filter(g => g.type && g.type.startsWith("brand gap"));
  const th = D.themes.length ? D.themes : [];
  $("#s-overview").innerHTML = `<div class="grid3">
   <div class="card"><h2>Corpus</h2><div class="big">${A.length}</div><div class="mute">articles from ${brands.length} brands</div>
    <table style="margin-top:8px">${brands.map(b => `<tr><td>${esc(b)}</td><td class="n">${nb[b]||0}</td></tr>`).join("")}</table></div>
   <div class="card"><h2>Cannibalization</h2><div class="big hl" id="ov-pairs"></div><div class="mute">pairs of articles from different brands that answer the same reader question (similarity ≥ threshold)</div>
"""

**Gabarit de la page (5/11)**.

In [ ]:
PAGE_5 = r"""
    <p>Default threshold <b>${D.t0}</b> (a simple choice, see the Threshold tab). Move the slider to see what changes.</p><div class="big" id="ov-same" style="margin-top:8px"></div><div class="mute">similar pairs inside a single brand (two very close articles of the same brand)</div><br><span class="link" onclick="cannMode='cross'; matrixSel=null; tab('cann')">open the brand matrix →</span> · <span class="link" onclick="cannMode='same'; matrixSel=null; tab('cann')">open the within-brand list →</span></div>
   <div class="card"><h2>White space</h2><div class="big" style="color:var(--gap)">${mw.length}</div><div class="mute">Mars-wide gaps: no brand covers the cell</div>
    <p>${mw.map(g => `<span class="tag red">${esc(g.cell)}</span>`).join(" ")}</p><div class="mute">+ ${bg.length} brand gaps inside each brand's own scope</div><span class="link" onclick="tab('white')">open the Mars grid →</span></div></div>
   <div class="card"><h2>Closest pairs at the current threshold</h2><div id="ov-top"></div></div>
   <div class="card"><h2>The ${th.length} themes found by K-means</h2><div class="scroll" style="max-height:none"><table><tr><th class="n">#</th><th>theme</th><th>family</th><th class="n">articles</th><th>most central article</th></tr>
   ${th.map(t => `<tr><td class="n">${t.theme}</td><td><b>${esc(t.name)}</b></td><td>${esc(t.family)}</td><td class="n">${t.n_articles}</td><td class="mute">${esc(t.representative_title)}</td></tr>`).join("")}</table></div></div>
   <div class="card"><h2>How to read this page</h2><ul class="mute" style="margin:0">
   <li><b>Semantic map</b>: every point is an article; close points = similar meaning. Colour by theme, family, brand or a Mars axis.</li>
   <li><b>Cannibalization</b>: brand × brand matrix and the list of competing pairs, with the paragraph overlap as evidence; switch to <i>Within a brand</i> for the similar pairs of one brand.</li>
   <li><b>White space</b>: pet type × food type × life stage per brand; red = nobody writes there, grey = outside the brand's positioning.</li>
   <li><b>Threshold</b>: how many pairs each threshold value keeps, and why the default value.</li></ul></div>`;
}
function ovPairs(){ const e = $("#ov-pairs"); if (e) e.textContent = cross().length; const s = $("#ov-same"); if (s) s.textContent = same().length; ovTop(); }
function ovTop(){ const e = $("#ov-top"); if (!e) return;
  const top5 = pool => { const r = pool.slice().sort((p,q) => q[2]-p[2]).slice(0,5);
    return r.length ? `<table><tr><th class="n">sim.</th><th>article A</th><th class="nw">brand</th><th>article B</th><th class="nw">brand</th></tr>${r.map(p => `<tr><td class="n"><b>${pct(p[2],1)}</b></td><td>${T_(p[0])}</td><td class="nw">${esc(A[p[0]].brand)}</td><td>${T_(p[1])}</td><td class="nw">${esc(A[p[1]].brand)}</td></tr>`).join("")}</table>` : `<p class="mute">None at threshold ${T.toFixed(3)}.</p>`; };
  e.innerHTML = `<div class="grid2" style="grid-template-columns:1fr 1fr"><div><h3 style="font-size:14px;margin:0 0 4px">Between brands</h3>${top5(cross())}</div><div><h3 style="font-size:14px;margin:0 0 4px">Within a brand</h3>${top5(same())}</div></div>
   <p class="mute"><span class="link" onclick="cannMode='cross'; matrixSel=null; tab('cann')">all pairs between brands →</span> · <span class="link" onclick="cannMode='same'; matrixSel=null; tab('cann')">all pairs within a brand →</span></p>`;
}
// ---------- map (SVG)
const COLOURS = [["theme","Theme (K-means)"],["family","Mars family (theme)"],["article_family","Mars family (article)"],["brand","Brand"],
  ["pet_type","Pet type"],["food_type","Food type"],["life_stage","Life stage"],["health_need","Health needs"],["health_subtheme","Health sub-theme"],["language","Language"]];
const GROUPS = [["Our classification", ["theme","family","article_family"]], ["Mars axes", ["pet_type","food_type","life_stage","health_need","health_subtheme"]], ["Other", ["brand","language"]]];
$("#colour").innerHTML = GROUPS.map(([g, ks]) => `<optgroup label="${g}">` + COLOURS.filter(([k]) => ks.includes(k) && k in A[0]).map(([k,l]) => `<option value="${k}">${l}</option>`).join("") + "</optgroup>").join("");
const svg = $("#map"), NS = "http://www.w3.org/2000/svg";
const xs = A.map(a => a.x), ys = A.map(a => a.y);
const X0 = Math.min(...xs), X1 = Math.max(...xs), Y0 = Math.min(...ys), Y1 = Math.max(...ys);
let view = null, VIEW0 = null;
"""

**Gabarit de la page (6/11)**.

In [ ]:
PAGE_6 = r"""
function initView(){ const W = svg.clientWidth || 900, H = svg.clientHeight || 640, rx = X1-X0+1, ry = Y1-Y0+1;
  const w = Math.max(rx, ry*W/H), h = w*H/W; VIEW0 = {x: (X0+X1)/2 - w/2, y: -(Y0+Y1)/2 - h/2, w}; view = {...VIEW0}; }
function main(v){ return String(v||"").split(" + ")[0]; }
function key(a, c){ if (c==="theme") return a.theme; if (c==="food_type"||c==="life_stage") return main(a[c]); return a[c] || "(none)"; }
function colourMap(c){
  const vals = [...new Set(A.map(a => key(a,c)))].sort((p,q) => (typeof p==="number" ? p-q : String(p).localeCompare(String(q))));
  const m = {}; vals.forEach((v,i) => m[v] = FIX[v] || PAL[i % PAL.length]); return [vals, m]; }
function label(c, v){ if (c==="theme"){ const a = A.find(x => x.theme===v); return v + " · " + (a ? a.theme_name : ""); } return v; }
function draw(){
  const c = $("#colour").value, [vals, m] = colourMap(c);
  if (!view) initView();
  const W = svg.clientWidth || 900, H = svg.clientHeight || 640, r = view.w / W;
  const q = ($("#search").value || "").trim().toLowerCase().split(/[\s,]+/).filter(t => t.length > 1);
  const hit = id => q.length && q.every(t => A[id].title.toLowerCase().includes(t));
  let s = "";
  if ($("#links").checked) cross().forEach(([a,b,sim]) => { const A1 = A[a], B1 = A[b];
    if (hidden.has(key(A1,c)) && hidden.has(key(B1,c))) return;
    s += `<line x1="${A1.x}" y1="${-A1.y}" x2="${B1.x}" y2="${-B1.y}" stroke="#d03b3b" stroke-opacity="${0.25 + (sim - D.tmin)*3}" stroke-width="${1.2*r}"/>`; });
  let nf = 0;
  A.forEach(a => { const k = key(a,c); if (hidden.has(k)) return; const h = hit(a.id); if (h) nf++;
    s += `<circle data-id="${a.id}" cx="${a.x}" cy="${-a.y}" r="${(sel===a.id ? 7 : 4.2)*r}" fill="${m[k]}" fill-opacity="${q.length && !h ? 0.15 : 0.9}" stroke="${sel===a.id||h ? "#111" : "#fff"}" stroke-width="${(sel===a.id||h ? 2 : 0.7)*r}"/>`; });
  if (sel !== null) cross().filter(p => p[0]===sel || p[1]===sel).forEach(p => { const o = A[p[0]===sel ? p[1] : p[0]];
    s += `<circle cx="${o.x}" cy="${-o.y}" r="${9*r}" fill="none" stroke="#d03b3b" stroke-width="${2.5*r}"/>`; });
  if ($("#labels").checked && (c==="theme" || c==="family")) vals.filter(v => !hidden.has(v)).forEach(v => {
    const g = A.filter(a => key(a,c)===v); if (g.length < 3) return;
    const mx = g.map(a => a.x).sort((p,q) => p-q)[g.length>>1], my = g.map(a => a.y).sort((p,q) => p-q)[g.length>>1];
    let t = c==="theme" ? v + " " + g[0].theme_name : String(v).toUpperCase(); if (t.length > 26) t = t.slice(0, 25) + "…";
    s += `<text x="${mx}" y="${-my}" font-size="${(c==="theme" ? 12 : 13)*r}" font-weight="700" text-anchor="middle" fill="${m[v]}" stroke="#fff" stroke-width="${5*r}" stroke-opacity="0.9" paint-order="stroke" style="pointer-events:none">${esc(t)}</text>`; });
"""

**Gabarit de la page (7/11)**.

In [ ]:
PAGE_7 = r"""
  svg.innerHTML = s;
  svg.setAttribute("viewBox", `${view.x} ${view.y} ${view.w} ${view.w*H/W}`);
  $("#found").textContent = q.length ? nf + " articles found" : "";
  const n = {}; A.forEach(a => { const k = key(a,c); n[k] = (n[k]||0)+1; });
  $("#legend").innerHTML = vals.map(v => `<span data-v="${esc(v)}" class="${hidden.has(v) ? "off" : ""}" title="${esc(label(c,v))}"><i style="background:${m[v]}"></i><em>${esc(label(c,v))}</em> <b>${n[v]}</b></span>`).join("");
  document.querySelectorAll("#legend span").forEach(e => e.onclick = () => { let v = e.dataset.v; if (c==="theme") v = +v;
    hidden.has(v) ? hidden.delete(v) : hidden.add(v); draw(); });
}
$("#colour").onchange = () => { hidden.clear(); draw(); }; $("#links").onchange = draw; $("#labels").onchange = draw; $("#search").oninput = draw;
window.addEventListener("resize", () => { if ($("#s-map").classList.contains("on")) draw(); });
$("#zoomreset").onclick = () => { initView(); draw(); };
svg.addEventListener("wheel", e => { e.preventDefault(); const b = svg.getBoundingClientRect(), k = e.deltaY > 0 ? 1.15 : 1/1.15;
  const fx = (e.clientX-b.left)/b.width, fy = (e.clientY-b.top)/b.height, h = view.w*b.height/b.width;
  const cx = view.x + fx*view.w, cy = view.y + fy*h; view.w *= k; const h2 = view.w*b.height/b.width;
  view.x = cx - fx*view.w; view.y = cy - fy*h2; draw(); }, {passive:false});
let drag = null;
svg.addEventListener("pointerdown", e => { drag = {x:e.clientX, y:e.clientY, v:{...view}, moved:false}; });
window.addEventListener("pointermove", e => { if (!drag) { tipMove(e); return; } const b = svg.getBoundingClientRect(), k = view.w / b.width;
  const dx = e.clientX-drag.x, dy = e.clientY-drag.y; if (Math.abs(dx)+Math.abs(dy) > 3) drag.moved = true;
  view.x = drag.v.x - dx*k; view.y = drag.v.y - dy*k; if (drag.moved) draw(); });
window.addEventListener("pointerup", e => { if (drag && !drag.moved && e.target.dataset && e.target.dataset.id) card(+e.target.dataset.id); drag = null; });
const tip = $("#tip");
function tipMove(e){ const id = e.target && e.target.dataset ? e.target.dataset.id : undefined;
  if (id === undefined) { tip.style.display = "none"; return; } const a = A[+id];
  tip.innerHTML = `<b>${esc(a.title)}</b><br>${esc(a.brand)} · theme ${a.theme} · ${esc(a.pet_type)} · ${esc(a.food_type)} · ${esc(a.life_stage)}`;
  tip.style.display = "block"; tip.style.left = Math.min(e.clientX+14, innerWidth-360)+"px"; tip.style.top = (e.clientY+14)+"px"; }
// ---------- article card
const T_ = id => `<span class="link" onclick="card(${id})">${esc(A[id].title)}</span>`;
"""

**Gabarit de la page (8/11)**.

In [ ]:
PAGE_8 = r"""
function card(id){
  sel = id; const a = A[id]; if (!$("#s-map").classList.contains("on")) tab("map");
  const comp = cross().filter(p => p[0]===id || p[1]===id).sort((p,q) => q[2]-p[2]);
  const dup = same().filter(p => p[0]===id || p[1]===id).sort((p,q) => q[2]-p[2]);
  const other = p => p[0]===id ? p[1] : p[0];
  const nm = D.names[a.theme] || [];
  $("#panel").innerHTML = `<h2>${esc(a.title)}</h2>
   <p><span class="tag">${esc(a.brand)}</span><span class="tag">${esc(a.language)}</span> <a href="${esc(a.url)}" target="_blank" rel="noopener">open the article ↗</a></p>
   <h3 style="font-size:14px;margin:10px 0 4px">Competing articles of other brands <span class="tag ${comp.length ? "red" : "green"}">${comp.length}</span></h3>
   ${comp.length ? `<table><tr><th>article</th><th>brand</th><th class="n">sim.</th><th class="n">overlap</th></tr>${comp.map(p => `<tr><td>${T_(other(p))}</td><td>${esc(A[other(p)].brand)}</td><td class="n">${pct(p[2],1)}</td><td class="n">${pct(p[3],0)}</td></tr>`).join("")}</table>`
     : `<p class="mute">None at threshold ${T.toFixed(3)}.</p>`}
   ${dup.length ? `<details><summary>Same-brand similar pairs (${dup.length})</summary><table>${dup.map(p => `<tr><td>${T_(other(p))}</td><td class="n">${pct(p[2],1)}</td></tr>`).join("")}</table></details>` : ""}
   <h3 style="font-size:14px;margin:12px 0 4px">Classification</h3>
   <table class="kv"><tr><td>Theme</td><td><b>${a.theme} · ${esc(a.theme_name)}</b> <span class="mute">(rank ${a.rank_in_theme} in theme)</span></td></tr>
   <tr><td>Mars family</td><td>${esc(a.family)} <span class="mute">(article: ${esc(a.article_family)})</span></td></tr>
   <tr><td>Pet type</td><td>${esc(a.pet_type)}</td></tr><tr><td>Food type</td><td>${esc(a.food_type)}</td></tr>
   <tr><td>Life stage</td><td>${esc(a.life_stage)}</td></tr><tr><td>Health needs</td><td>${esc(a.health_need)}${a.health_subtheme ? " · " + esc(a.health_subtheme) : ""}</td></tr>
   <tr><td>Mars grid cells</td><td>${(a.cells||[]).map(c => `<span class="tag">${esc(c)}</span>`).join("")}</td></tr></table>
   ${nm.length ? `<details><summary>Why is the theme called “${esc(a.theme_name)}”?</summary><p class="mute">score = sim. with theme centre − 0.5 × sim. with nearest other theme + 0.15 × coverage</p>
    <table><tr><th>expression</th><th class="n">coverage</th><th class="n">sim.</th><th class="n">other</th><th class="n">score</th></tr>${nm.map(d => `<tr${d.chosen ? ' style="background:#fff7d6"' : ""}><td>${esc(d.expression)}${d.chosen ? " <b>← word " + d.chosen + "</b>" : ""}</td><td class="n">${Math.round(100*d.coverage)}%</td><td class="n">${(+d.sim_theme).toFixed(2)}</td><td class="n">${(+d.sim_neighbour).toFixed(2)}</td><td class="n">${(+d.score).toFixed(2)}</td></tr>`).join("")}</table></details>` : ""}`;
  draw();
}
// ---------- cannibalization tab
let cannMode = "cross";
function cann(){
  const B = D.brands, c = cross(), sm = same(), M = {}; B.forEach(x => B.forEach(y => M[x+"|"+y] = 0));
  c.forEach(([a,b]) => { M[A[a].brand+"|"+A[b].brand]++; M[A[b].brand+"|"+A[a].brand]++; });
  sm.forEach(([a]) => { M[A[a].brand+"|"+A[a].brand]++; });
"""

**Gabarit de la page (9/11)**.

In [ ]:
PAGE_9 = r"""
  const mx = Math.max(1, ...B.flatMap(x => B.filter(y => y!==x).map(y => M[x+"|"+y]))), mxd = Math.max(1, ...B.map(x => M[x+"|"+x]));
  const pool = cannMode==="cross" ? c : sm;
  const rows = pool.filter(([a,b]) => !matrixSel || [A[a].brand, A[b].brand].sort().join("|") === matrixSel).sort((p,q) => q[2]-p[2]);
  const q = ($("#ptq") ? $("#ptq").value : "").toLowerCase();
  const shown = rows.filter(([a,b]) => !q || (A[a].title + " " + A[b].title).toLowerCase().includes(q));
  const prev = $("#ptq") ? $("#ptq").value : "";
  $("#s-cann").innerHTML = `<div class="tools"><button class="mode ${cannMode==="cross" ? "on" : ""}" data-m="cross">Between brands — cannibalization (${c.length})</button>
   <button class="mode ${cannMode==="same" ? "on" : ""}" data-m="same">Within a brand — similar pairs (${sm.length})</button></div>
   <div class="grid2"><div class="card"><h2>Pairs at threshold ${T.toFixed(3)}: brand × brand</h2>
   <table class="heat"><tr><th></th>${B.map(b => `<th style="text-align:center">${esc(b)}</th>`).join("")}</tr>
   ${B.map(x => `<tr><th>${esc(x)}</th>${B.map(y => { const k = [x,y].sort().join("|"), v = M[x+"|"+y];
     if (x===y) { const t = v/mxd; return `<td class="c ${matrixSel===k ? "sel" : ""}" data-k="${k}" title="similar pairs within ${esc(x)}" style="background:rgba(124,58,237,${(0.06+0.8*t).toFixed(2)});color:${t>0.5 ? "#fff" : "#1f2933"}">${v}</td>`; }
     const t = v/mx; return `<td class="c ${matrixSel===k ? "sel" : ""}" data-k="${k}" style="background:rgba(37,99,235,${(0.06+0.9*t).toFixed(2)});color:${t>0.5 ? "#fff" : "#1f2933"}">${v}</td>`; }).join("")}</tr>`).join("")}</table>
   ${matrixSel ? `<p class="mute"><span class="link" id="clr">show all</span></p>` : ""}</div>
   <div class="card"><h2>Reading</h2><ul class="mute" style="margin:0"><li><b>similarity</b>: whole articles (bge-m3), the score used for the threshold.</li>
   <li><b>overlap</b>: paragraph by paragraph (best match of each paragraph, both ways); close to 1 = same content, not only the same topic.</li>
   <li><b>same cell</b>: both articles in the same Mars grid cell (pet × food × stage) = direct competition for the same need.</li>
   <li><b>Between brands</b> = cannibalization: two Mars brands compete for the same reader question.</li>
   <li><b>Within a brand</b> = similar pairs: one brand has two very close articles.</li></ul></div></div>
   <div class="card"><h2>${shown.length} ${cannMode==="cross" ? "competing pairs between brands" : "similar pairs within a brand"}${matrixSel ? " · " + esc(matrixSel.split("|")[0]===matrixSel.split("|")[1] ? matrixSel.split("|")[0] : matrixSel.replace("|"," – ")) : ""}</h2>
   <input type="search" id="ptq" placeholder="Filter by title words" value="${esc(prev)}" style="margin-bottom:8px">
   <div class="scroll"><table><tr><th class="n">sim.</th><th class="n">overlap</th><th>article A</th><th class="nw">brand</th><th>article B</th><th class="nw">brand</th>${cannMode==="cross" ? '<th class="nw">same cell</th>' : ""}</tr>
   ${shown.slice(0,400).map(([a,b,s,o]) => { const sc = (A[a].cells||[]).some(x => (A[b].cells||[]).includes(x));
     const last = cannMode==="cross" ? `<td class="nw">${sc ? '<span class="tag red">yes</span>' : '<span class="tag">no</span>'}</td>` : "";
     return `<tr><td class="n"><b>${pct(s,1)}</b></td><td class="n">${pct(o,0)}</td><td>${T_(a)}</td><td class="nw">${esc(A[a].brand)}</td><td>${T_(b)}</td><td class="nw">${esc(A[b].brand)}</td>${last}</tr>`; }).join("")}</table></div>
   ${shown.length > 400 ? `<p class="mute">First 400 shown; raise the threshold or filter.</p>` : ""}</div>`;
  document.querySelectorAll("#s-cann button.mode").forEach(e => e.onclick = () => { cannMode = e.dataset.m; matrixSel = null; cann(); });
  document.querySelectorAll(".heat td.c").forEach(e => e.onclick = () => { const k = e.dataset.k, [x,y] = k.split("|");
"""

**Gabarit de la page (10/11)**.

In [ ]:
PAGE_10 = r"""
    if (matrixSel===k) matrixSel = null; else { matrixSel = k; cannMode = x===y ? "same" : "cross"; } cann(); });
  if ($("#clr")) $("#clr").onclick = () => { matrixSel = null; cann(); };
  const box = $("#ptq"); box.oninput = () => { cann(); const b2 = $("#ptq"); b2.focus(); b2.setSelectionRange(b2.value.length, b2.value.length); };
}
// ---------- white space tab
let cellSel = null;
function white(){
  const B = D.brands, G = D.grid, mx = Math.max(1, ...G.flatMap(r => B.map(b => +r[b])));
  const gapType = {}; D.gaps.forEach(g => { gapType[g.cell + "|" + g.brand] = g.type; });
  const list = cellSel ? A.filter(a => (a.cells||[]).includes(cellSel.cell) && (cellSel.brand==="TOTAL" || a.brand===cellSel.brand)) : [];
  $("#s-white").innerHTML = `<div class="grid2"><div class="card"><h2>Mars grid: articles per brand (pet type × food type × life stage)</h2>
   <div class="scroll" style="max-height:none"><table class="heat"><tr><th>pet</th><th>food type</th><th>life stage</th>${B.map(b => `<th style="text-align:center">${esc(b)}</th>`).join("")}<th style="text-align:center">all</th></tr>
   ${G.map(r => { const cell = `${r.species} · ${r.food_type} · ${r.life_stage}`, tot = +r.TOTAL;
     return `<tr><td>${esc(r.species)}</td><td>${esc(r.food_type)}</td><td>${esc(r.life_stage)}</td>${B.map(b => { const v = +r[b];
       const out = !(D.scope[b]||[]).includes(r.species), g = gapType[cell+"|"+b];
       if (out) return `<td class="c" style="background:#f1f3f5;color:#adb5bd" title="outside ${esc(b)}'s positioning">–</td>`;
       const st = v===0 ? (g ? "background:var(--gapbg);color:var(--gap);outline:2px solid var(--gap);outline-offset:-2px" : "background:#fff;color:#adb5bd")
         : `background:rgba(37,99,235,${(0.08+0.85*v/mx).toFixed(2)});color:${v/mx>0.5 ? "#fff" : "#1f2933"}`;
       return `<td class="c ${cellSel && cellSel.cell===cell && cellSel.brand===b ? "sel" : ""}" data-c="${esc(cell)}" data-b="${esc(b)}" style="${st}" title="${g ? esc(g) : ""}">${v}</td>`; }).join("")}
     <td class="c" data-c="${esc(cell)}" data-b="TOTAL" style="${tot===0 ? "background:var(--gap);color:#fff" : tot<=2 ? "background:#fff4e2;color:#b45309" : "background:#f8f9fa"}">${tot}</td></tr>`; }).join("")}</table></div>
   <p class="mute"><span class="tag red">red 0</span> gap for a brand inside its own scope · <span class="tag" style="background:var(--gap);color:#fff">red total</span> nobody covers the cell · <span class="tag" style="background:#fff4e2;color:#b45309">orange total</span> thin (1–2 articles) · grey “–” the brand does not write about this pet type (positioning, not a gap). Click a number to list its articles.</p></div>
   <div><div class="card"><h2>${cellSel ? esc(cellSel.cell) + " · " + esc(cellSel.brand) : "Articles of a cell"}</h2>
   ${cellSel ? (list.length ? `<ol>${list.map(a => `<li>${T_(a.id)} <span class="mute">(${esc(a.brand)})</span></li>`).join("")}</ol>` : `<p class="mute">No article: white space.</p>`) : `<p class="mute">Click a number in the grid.</p>`}</div>
   <div class="card"><h2>Gaps found</h2>${["Mars-wide gap","thin coverage","brand gap","brand gap (theme)"].map(t => { const g = D.gaps.filter(x => x.type===t); if (!g.length) return "";
     return `<details ${t==="Mars-wide gap" ? "open" : ""}><summary>${esc(t)} (${g.length})</summary><table><tr><th>cell / theme</th><th>brand</th><th class="n">sisters</th></tr>${g.map(x => `<tr><td>${esc(x.cell)}</td><td>${esc(x.brand)}</td><td class="n">${esc(x.sister_brand_articles)}</td></tr>`).join("")}</table></details>`; }).join("")}</div></div></div>`;
  document.querySelectorAll("#s-white td.c[data-c]").forEach(e => e.onclick = () => { cellSel = {cell: e.dataset.c, brand: e.dataset.b}; white(); });
}
// ---------- threshold tab (SVG chart): number of cross-brand pairs kept for every threshold
"""

**Gabarit de la page (11/11)** : fin du JavaScript.

In [ ]:
PAGE_11 = r"""
function drawThr(){
  const ts = []; for (let t = D.tmin; t <= D.tmax + 0.001; t += 0.005) ts.push(+t.toFixed(3));
  const cnt = t => P.filter(p => p[4] && p[2] >= t).length, cmax = cnt(D.tmin);
  const W = 900, H = 330, L = 50, R = 60, Tp = 20, Bt = 40, x = t => L + (t-D.tmin)/(D.tmax-D.tmin)*(W-L-R), y2 = v => Tp + (1-v/cmax)*(H-Tp-Bt);
  let s = `<svg viewBox="0 0 ${W} ${H}" class="chart">`;
  for (let t = D.tmin; t <= D.tmax + 0.001; t += 0.05) s += `<text x="${x(t)}" y="${H-Bt+18}" font-size="11" text-anchor="middle" fill="#5f6b7a">${t.toFixed(2)}</text>`;
  s += `<text x="${(W)/2}" y="${H-4}" font-size="12" text-anchor="middle" fill="#5f6b7a">similarity threshold</text>`;
  s += `<text x="${W-R+8}" y="${Tp+4}" font-size="11" fill="#5f6b7a">${cmax} pairs</text><text x="${W-R+8}" y="${H-Bt}" font-size="11" fill="#5f6b7a">0</text>`;
  s += `<polygon fill="#dbe7fb" stroke="#2563eb" stroke-width="2" points="${x(D.tmin)},${y2(0)} ${ts.map(t => x(t)+","+y2(cnt(t))).join(" ")} ${x(D.tmax)},${y2(0)}"/>`;
  s += `<line x1="${x(T)}" x2="${x(T)}" y1="${Tp}" y2="${H-Bt}" stroke="#1f2933" stroke-dasharray="4 3"/><text x="${x(T)+5}" y="${Tp+12}" font-size="12" font-weight="700">${T.toFixed(3)}</text>`;
  s += `<line x1="${x(D.t0)}" x2="${x(D.t0)}" y1="${Tp}" y2="${H-Bt}" stroke="#d03b3b" stroke-width="1.5"/></svg>`;
  $("#s-thr").innerHTML = `<div class="grid2"><div class="card"><h2>Cross-brand pairs kept, for every threshold</h2>${s}
   <p class="mute">Blue area: number of pairs of articles from different brands whose similarity is at least the threshold. Red line: default threshold. Dashed line: current threshold.</p></div>
   <div class="card"><h2>At ${T.toFixed(3)}</h2><table class="kv"><tr><td>cross-brand pairs</td><td><b>${cross().length}</b></td></tr><tr><td>same-brand similar pairs</td><td>${same().length}</td></tr>
   <tr><td>at 0.80 / 0.85</td><td>${cnt(0.8)} / ${cnt(0.85)} pairs</td></tr></table>
   <div class="note">Why not a fixed 0.50? All articles are about pets: the median similarity of two articles is about 0.57, so 0.50 keeps almost every pair.</div>
   <div class="note">Why ${D.t0}? It keeps the most similar 0.1 % of the pairs between brands: clear cases without drowning the reading. It is a reading choice, not a measured optimum: move the slider to see how the list changes.</div>
   <p class="mute">A high similarity does not prove the same search intent: read the two articles before acting.</p></div></div>`;
}
// ---------- wiring
function refresh(){ kpi(); ovPairs(); const k = document.querySelector("nav button.on").dataset.k;
  if (k==="map") { draw(); if (sel!==null) card(sel); } if (k==="cann") cann(); if (k==="thr") drawThr(); }
thr.oninput = () => { T = +thr.value; refresh(); };
$("#reset").onclick = () => { T = D.t0; thr.value = T; refresh(); };
function fit(){ try { const f = window.frameElement; if (f) f.style.height = (document.documentElement.scrollHeight + 30) + "px"; } catch (e) {} }
overview(); cann(); white(); kpi(); ovPairs(); tab("overview");
const _tab = tab; tab = k => { _tab(k); if (k==="cann") cann(); if (k==="white") white(); fit(); };
</script></body></html>
"""

**Traduction FR / EN** : dictionnaire des textes de l'interface (clé = texte anglais, valeur = texte français), puis quelques règles pour les textes qui contiennent un nombre. Les données (titres, noms de thèmes) ne sont pas traduites.

In [ ]:
FR_1 = {
    "Mars Petcare — content map, cannibalization and white space": "Mars Petcare — carte des contenus, cannibalisation et espaces vides",
    "5 brands · articles classified by meaning (bge-m3) · themes (K-means) and the 4 Mars axes · one threshold drives every view": "5 marques · articles classés par le sens (bge-m3) · thèmes (K-means) et les 4 axes Mars · un seul seuil pilote toutes les vues",
    "Overview": "Vue d'ensemble",
    "Semantic map": "Carte sémantique",
    "Cannibalization": "Cannibalisation",
    "White space": "Espaces vides",
    "Threshold": "Seuil",
    "Cannibalization threshold": "Seuil de cannibalisation",
    "reset to default": "valeur par défaut",
    "cross-brand pairs": "paires entre marques",
    "articles involved": "articles concernés",
    "same-brand similar pairs": "paires proches dans une même marque",
    "Corpus": "Corpus",
    "pairs of articles from different brands that answer the same reader question (similarity ≥ threshold)": "paires d'articles de marques différentes qui répondent à la même question du lecteur (similarité ≥ seuil)",
    "Default threshold": "Seuil par défaut",
    "(a simple choice, see the Threshold tab). Move the slider to see what changes.": "(un choix simple, voir l'onglet Seuil). Déplacer le curseur pour voir ce qui change.",
    "open the brand matrix →": "ouvrir la matrice des marques →",
    "Mars-wide gaps: no brand covers the cell": "vides à l'échelle de Mars : aucune marque ne couvre la case",
    "open the Mars grid →": "ouvrir la grille Mars →",
    "#": "#",
    "theme": "thème",
    "family": "famille",
}

FR_2 = {
    "articles": "articles",
    "most central article": "article le plus central",
    "How to read this page": "Comment lire cette page",
    ": every point is an article; close points = similar meaning. Colour by theme, family, brand or a Mars axis.": " : chaque point est un article ; des points proches = un sens proche. Couleur par thème, famille, marque ou axe Mars.",
    ": brand × brand matrix and the list of competing pairs, with the paragraph overlap as evidence; switch to": " : matrice marque × marque et liste des paires concurrentes, avec le recouvrement des paragraphes comme preuve ; passer à",
    "Within a brand": "Dans une même marque",
    "for the similar pairs of one brand.": "pour les paires proches d'une marque.",
    ": pet type × food type × life stage per brand; red = nobody writes there, grey = outside the brand's positioning.": " : animal × type d'aliment × stade de vie par marque ; rouge = personne n'écrit là, gris = hors du positionnement de la marque.",
    ": how many pairs each threshold value keeps, and why the default value.": " : combien de paires chaque valeur du seuil garde, et pourquoi la valeur par défaut.",
    "Colour": "Couleur",
    "Theme (K-means)": "Thème (K-means)",
    "Mars family (theme)": "Famille Mars (thème)",
    "Mars family (article)": "Famille Mars (article)",
    "Pet type": "Animal",
    "Food type": "Type d'aliment",
    "Life stage": "Stade de vie",
    "Health needs": "Besoin santé",
    "Health sub-theme": "Sous-thème santé",
    "Brand": "Marque",
    "Language": "Langue",
    "cannibalization links": "liens de cannibalisation",
    "labels": "étiquettes",
}

FR_3 = {
    "Search titles (e.g. dental, senior cat, wet food)": "Chercher dans les titres (ex. dental, senior cat, wet food)",
    "reset zoom": "réinitialiser le zoom",
    "Wheel: zoom · drag: pan · click a point: article card. Red lines join articles of different brands with similarity ≥ threshold. UMAP axes have no meaning; only distances count.": "Molette : zoom · glisser : déplacer · clic sur un point : fiche de l'article. Les lignes rouges relient des articles de marques différentes de similarité ≥ seuil. Les axes d'UMAP n'ont pas de signification ; seules les distances comptent.",
    "Article card": "Fiche article",
    "Click a point on the map, or a title anywhere in the page.": "Cliquer sur un point de la carte, ou sur un titre n'importe où dans la page.",
    "Reading": "Lecture",
    "similarity": "similarité",
    ": whole articles (bge-m3), the score used for the threshold.": " : articles entiers (bge-m3), le score utilisé pour le seuil.",
    "overlap": "recouvrement",
    ": paragraph by paragraph (best match of each paragraph, both ways); close to 1 = same content, not only the same topic.": " : paragraphe par paragraphe (meilleure correspondance de chaque paragraphe, dans les deux sens) ; proche de 1 = même contenu, pas seulement le même sujet.",
    "same cell": "même case",
    ": both articles in the same Mars grid cell (pet × food × stage) = direct competition for the same need.": " : les deux articles sont dans la même case de la grille Mars (animal × aliment × stade) = concurrence directe sur le même besoin.",
    "Between brands": "Entre marques",
    "= cannibalization: two Mars brands compete for the same reader question.": "= cannibalisation : deux marques Mars se disputent la même question de lecteur.",
    "= similar pairs: one brand has two very close articles.": "= paires proches : une marque a deux articles très proches.",
    "Filter by title words": "Filtrer par mots du titre",
    "sim.": "sim.",
    "article A": "article A",
    "brand": "marque",
    "article B": "article B",
    "First 400 shown; raise the threshold or filter.": "400 premières affichées ; augmenter le seuil ou filtrer.",
    "show all": "tout afficher",
}

FR_4 = {
    "Mars grid: articles per brand (pet type × food type × life stage)": "Grille Mars : articles par marque (animal × type d'aliment × stade de vie)",
    "pet": "animal",
    "food type": "type d'aliment",
    "life stage": "stade de vie",
    "all": "total",
    "brand gap": "lacune de marque",
    "red 0": "0 rouge",
    "gap for a brand inside its own scope ·": "lacune d'une marque dans son propre périmètre ·",
    "red total": "total rouge",
    "nobody covers the cell ·": "personne ne couvre la case ·",
    "orange total": "total orange",
    "thin (1–2 articles) · grey “–” the brand does not write about this pet type (positioning, not a gap). Click a number to list its articles.": "faible (1–2 articles) · « – » gris : la marque ne publie pas sur cet animal (positionnement, pas une lacune). Cliquer sur un nombre pour lister ses articles.",
    "Articles of a cell": "Articles d'une case",
    "Click a number in the grid.": "Cliquer sur un nombre de la grille.",
    "Gaps found": "Lacunes trouvées",
    "cell / theme": "case / thème",
    "sisters": "autres marques",
    "open the article ↗": "ouvrir l'article ↗",
    "Competing articles of other brands": "Articles concurrents des autres marques",
    "article": "article",
    "Classification": "Classification",
    "Theme": "Thème",
}

FR_5 = {
    "Mars family": "Famille Mars",
    "Mars grid cells": "Cases de la grille Mars",
    "No article: white space.": "Aucun article : espace vide.",
    "Cross-brand pairs kept, for every threshold": "Paires entre marques gardées, pour chaque seuil",
    "similarity threshold": "seuil de similarité",
    "Blue area: number of pairs of articles from different brands whose similarity is at least the threshold. Red line: default threshold. Dashed line: current threshold.": "Zone bleue : nombre de paires d'articles de marques différentes dont la similarité atteint au moins le seuil. Ligne rouge : seuil par défaut. Ligne en pointillés : seuil actuel.",
    "at 0.80 / 0.85": "à 0,80 / 0,85",
    "Why not a fixed 0.50? All articles are about pets: the median similarity of two articles is about 0.57, so 0.50 keeps almost every pair.": "Pourquoi pas un seuil fixe à 0,50 ? Tous les articles parlent d'animaux : la similarité médiane entre deux articles est d'environ 0,57, donc 0,50 garde presque toutes les paires.",
    "A high similarity does not prove the same search intent: read the two articles before acting.": "Une similarité élevée ne prouve pas la même intention de recherche : lire les deux articles avant d'agir.",
    "similar pairs inside a single brand (two very close articles of the same brand)": "paires proches à l'intérieur d'une même marque (deux articles très proches d'une même marque)",
    "open the within-brand list →": "ouvrir la liste dans une même marque →",
    "Closest pairs at the current threshold": "Paires les plus proches au seuil actuel",
    "all pairs between brands →": "toutes les paires entre marques →",
    "all pairs within a brand →": "toutes les paires dans une même marque →",
    "Our classification": "Notre classification",
    "Mars axes": "Axes Mars",
    "Other": "Autre",
    "yes": "oui",
    "no": "non",
}

FR_EXACT = {**FR_1, **FR_2, **FR_3, **FR_4, **FR_5}

FR_REGEX = [   # [motif, remplacement] : $1, $2... sont les nombres ou noms repris du texte anglais
    ["^articles from (\\d+) brands$", "articles de $1 marques"],
    ["^\\+ (\\d+) brand gaps inside each brand's own scope$", "+ $1 lacunes de marque dans le périmètre de chaque marque"],
    ["^The (\\d+) themes found by K-means$", "Les $1 thèmes trouvés par K-means"],
    ["^Default threshold$", "Seuil par défaut"],
    ["^Between brands — cannibalization \\((\\d+)\\)$", "Entre marques — cannibalisation ($1)"],
    ["^Within a brand — similar pairs \\((\\d+)\\)$", "Dans une même marque — paires proches ($1)"],
    ["^Pairs at threshold ([\\d.]+): brand × brand$", "Paires au seuil $1 : marque × marque"],
    ["^similar pairs within (\\w+)$", "paires proches dans $1"],
    ["^(\\d+) competing pairs between brands(.*)$", "$1 paires concurrentes entre marques$2"],
    ["^(\\d+) similar pairs within a brand$", "$1 paires proches dans une même marque"],
    ["^(\\d+) articles found$", "$1 articles trouvés"],
    ["^([^·]+) · theme (\\d+) · (.*)$", "$1 · thème $2 · $3"],
    ["^\\(rank (\\d+) in theme\\)$", "(rang $1 dans le thème)"],
    ["^\\(article: (.*)\\)$", "(article : $1)"],
    ["^Same-brand similar pairs \\((\\d+)\\)$", "Paires proches de la même marque ($1)"],
    ["^Mars-wide gap \\((\\d+)\\)$", "Vide à l'échelle de Mars ($1)"],
    ["^thin coverage \\((\\d+)\\)$", "Couverture faible ($1)"],
    ["^brand gap \\((\\d+)\\)$", "Lacune de marque ($1)"],
    ["^brand gap \\(theme\\) \\((\\d+)\\)$", "Lacune de marque (thème) ($1)"],
    ["^(\\d+) pairs$", "$1 paires"],
    ["^(\\d+) / (\\d+) pairs$", "$1 / $2 paires"],
    ["^At ([\\d.]+)$", "À $1"],
    ["^None at threshold ([\\d.]+)\\.$", "Aucune au seuil $1."],
    ["^outside (\\w+)'s positioning$", "hors du positionnement de $1"],
    ["^Why ([\\d.]+)\\? It keeps the most similar 0\\.1 % of the pairs between brands: clear cases without drowning the reading\\. It is a reading choice, not a measured optimum: move the slider to see how the list changes\\.$", "Pourquoi $1 ? Cela garde les 0,1 % de paires les plus proches entre marques : des cas nets sans noyer la lecture. C'est un choix de lecture, pas un optimum mesuré : déplacer le curseur pour voir comment la liste change."],
]

**Script de traduction** : à l'affichage, remplace les textes anglais de la page par leur traduction (et restaure l'anglais au clic sur EN). Il surveille aussi les parties de la page redessinées quand on clique ou qu'on déplace le curseur.

In [ ]:
JS_LANGUE = r"""let LANG = "fr";
const ORIG = new WeakMap(), ORIGA = new WeakMap();
function fr(v){ const k = v.trim(); if (!k) return v;
  if (FR_EXACT.has(k)) return v.replace(k, () => FR_EXACT.get(k));
  for (const [re, to] of FR_REGEX) if (re.test(k)) return v.replace(k, () => k.replace(re, to));
  return v; }
function trNode(n){
  if (n.nodeType === 3) { if (!ORIG.has(n)) ORIG.set(n, n.nodeValue); n.nodeValue = LANG === "fr" ? fr(ORIG.get(n)) : ORIG.get(n); return; }
  if (n.nodeType !== 1 || n.tagName === "SCRIPT" || n.tagName === "STYLE") return;
  for (const a of ["placeholder", "title", "label"]) if (n.hasAttribute(a)) {
    const m = ORIGA.get(n) || {}; if (!(a in m)) m[a] = n.getAttribute(a); ORIGA.set(n, m);
    n.setAttribute(a, LANG === "fr" ? fr(m[a]) : m[a]); }
  n.childNodes.forEach(trNode);
}
function setLang(l){ LANG = l; document.documentElement.lang = l;
  document.title = l === "fr" ? "Carte des contenus Mars" : "Mars content map";
  document.querySelectorAll("#langs button").forEach(b => b.classList.toggle("on", b.dataset.l === l)); trNode(document.body); }
new MutationObserver(ms => ms.forEach(m => m.addedNodes.forEach(trNode))).observe(document.body, {childList: true, subtree: true});
document.querySelectorAll("#langs button").forEach(b => b.onclick = () => setLang(b.dataset.l));
setLang("fr");"""

**Assemblage** : le gabarit reçoit les données (JSON) et le script de traduction, puis la page est enregistrée dans `outputs/`.

In [ ]:
PAGE = "".join([PAGE_1, PAGE_2, PAGE_3, PAGE_4, PAGE_5, PAGE_6, PAGE_7, PAGE_8, PAGE_9, PAGE_10, PAGE_11])
json_page = json.dumps(donnees_page, ensure_ascii=False, default=str).replace("</", "<\\/")   # JSON sûr dans une balise <script>
script_langue = ("<script>const FR_EXACT = new Map(Object.entries(" + json.dumps(FR_EXACT, ensure_ascii=False) + "));\n"
                 "const FR_REGEX = " + json.dumps(FR_REGEX, ensure_ascii=False) + ".map(([p, r]) => [new RegExp(p), r]);\n" + JS_LANGUE + "</script>")
page_tableau = PAGE.replace("__DATA__", json_page).replace("</body></html>", script_langue + "</body></html>")
with open(f"{OUT_DIR}/mars_tableau_de_bord.html", "w", encoding="utf-8") as f:
    f.write(page_tableau)
print(f"Tableau de bord enregistré : {OUT_DIR}/mars_tableau_de_bord.html ({len(page_tableau) / 1e6:.1f} Mo)")

**Zip de livraison** : le tableau de bord, la carte interactive, les tableaux de résultats et un `LISEZ-MOI.txt`. Les zips des étapes 1 et 2 restent des fichiers de travail (ils servent à relancer le pipeline).
Sur Colab le zip est téléchargé automatiquement (Colab efface `/content` en fin de session).

In [ ]:
FICHIERS_LIVRAISON = ["mars_tableau_de_bord.html", "carte_interactive.html", "rapport_cannibalisation.md", "articles_themes.csv",
                      "themes_resume.csv", "cannibalisation_paires.csv", "paires_proches_meme_marque.csv", "cannibalisation_matrice_marques.csv",
                      "espaces_vides_grille.csv", "espaces_vides_lacunes.csv", "espaces_vides_themes.csv", "sous_themes_sante.csv", "grille_mars.csv"]
LISEZ_MOI = f"""Mars Petcare - cartographie sémantique des contenus
==================================================
mars_tableau_de_bord.html  page unique (s'ouvre dans un navigateur, sans connexion) : vue d'ensemble, carte, cannibalisation,
                           espaces vides, seuil. Un curseur règle le seuil de cannibalisation (défaut {SEUIL}). Boutons FR / EN.
carte_interactive.html     carte 2D des {len(articles)} articles (UMAP) : clic = fiche de l'article, recherche, couleurs par thème / famille / axe Mars.
rapport_cannibalisation.md paires d'articles de marques différentes très proches (similarité >= {SEUIL}), groupe par groupe.
articles_themes.csv        une ligne par article : marque, thème, famille Mars, animal, type d'aliment, stade de vie, besoin santé, coordonnées.
themes_resume.csv          les {K} thèmes : nom, famille, taille, article le plus central, mots-clés.
cannibalisation_*.csv      paires entre marques (groupe, similarité, recouvrement des paragraphes) et matrice marque x marque.
paires_proches_meme_marque.csv  paires d'une MÊME marque très proches en sens (pas de la cannibalisation entre marques ; de vrais doublons sont rares).
espaces_vides_*.csv        articles par animal x aliment x stade et par marque ; liste des lacunes. 0 = piste à vérifier, pas une certitude.
sous_themes_sante.csv, grille_mars.csv   sous-thèmes santé ; grille animal x aliment x stade x santé.
Attention : une zone vide sur la carte 2D n'est pas une lacune de contenu, et une similarité élevée ne prouve pas la même intention de recherche.
"""
chemin_zip = f"{OUT_DIR}/mars_livraison.zip"
with zipfile.ZipFile(chemin_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for nom in FICHIERS_LIVRAISON:
        z.write(f"{OUT_DIR}/{nom}", nom)
    z.writestr("LISEZ-MOI.txt", LISEZ_MOI)
print(f"Livraison : {chemin_zip} ({os.path.getsize(chemin_zip) / 1e6:.1f} Mo) :", zipfile.ZipFile(chemin_zip).namelist())

if IN_COLAB:
    from google.colab import files
    files.download(chemin_zip)
from IPython.display import HTML, display
display(HTML('<iframe srcdoc="' + html.escape(page_tableau, quote=True) + '" style="width:100%; height:1500px; border:0"></iframe>'))